相関解析

In [ ]:
pip install pandas openpyxl matplotlib

In [ ]:
import io
import re
import itertools
import math
from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm

import ipywidgets as widgets
from IPython.display import display, clear_output

from openpyxl import load_workbook
from openpyxl.drawing.image import Image as XLImage


# ============================================================
# 0) 固定設定
# ============================================================
OUTPUT_ROOT = Path.cwd() / "data" / "export"

SHEET_PLOTS = "plots"
SHEET_SUMMARY = "summary"
SHEET_GROUP_STATS = "group_stats"
SHEET_OUTLIERS = "outliers"
SHEET_SAMPLE_METRICS = "sample_metrics"
SHEET_METRIC_DEFINITIONS = "metric_definitions"

OUT_SUFFIX = "_with_QCplots"

REGRESSION_METHODS_ALL = ["OLS", "Deming", "TheilSen", "PassingBablok"]

REGRESSION_LABELS = {
    "OLS": "OLS（最小二乗）",
    "Deming": "Deming（両軸誤差）",
    "TheilSen": "Theil-Sen（ロバスト）",
    "PassingBablok": "Passing-Bablok（ノンパラメトリック）"
}


# ============================================================
# 1) 日本語フォント
# ============================================================
def setup_japanese_font():
    candidates = [
        "IPAexGothic", "IPAPGothic",
        "Noto Sans CJK JP", "Noto Sans JP",
        "Yu Gothic", "YuGothic",
        "Meiryo", "MS Gothic",
        "Hiragino Sans", "TakaoGothic"
    ]

    available = {f.name for f in fm.fontManager.ttflist}
    chosen = None

    for c in candidates:
        if c in available:
            chosen = c
            break

    if chosen is None:
        for name in sorted(available):
            if any(k in name for k in ["IPA", "Noto", "Gothic", "Meiryo", "Hiragino", "ゴシック", "明朝"]):
                chosen = name
                break

    if chosen:
        plt.rcParams["font.family"] = chosen

    plt.rcParams["axes.unicode_minus"] = False


setup_japanese_font()


# ============================================================
# 2) 列推定
# ============================================================
ID_COL_CANDIDATES = ["検体ID", "SampleID", "Sample ID", "ID"]
GROUP_COL_CANDIDATES = ["種別", "Group", "Type", "分類"]
VALUE_PREFIXES = ("処方",)


def pick_col(df, candidates, default=None):
    for c in candidates:
        if c in df.columns:
            return c
    return default


def normalize_group_col(df, group_col):
    if group_col is None:
        return None

    if group_col not in df.columns:
        return None

    s = df[group_col]

    if s.notna().sum() == 0:
        return None

    nunique = s.nunique(dropna=True)

    if nunique <= 1:
        return None

    if pd.api.types.is_numeric_dtype(s):
        n = s.notna().sum()
        if nunique > min(20, max(5, n // 3)):
            return None

    return group_col


def detect_value_cols(df, id_col, group_col, prefixes=("処方",)):
    cols = [c for c in df.columns if any(str(c).startswith(p) for p in prefixes)]

    if cols:
        return cols

    numeric_cols = df.select_dtypes(include="number").columns.tolist()

    exclude_cols = [id_col]
    if group_col:
        exclude_cols.append(group_col)

    return [c for c in numeric_cols if c not in exclude_cols]


def safe_name(s):
    s = str(s)
    return re.sub(r'[\\/:*?"<>|]', "_", s)


# ============================================================
# 3) FileUpload対応
# ============================================================
def _bytes_from_uploaded_content(content):
    if content is None:
        return None

    if isinstance(content, (bytes, bytearray)):
        return bytes(content)

    if hasattr(content, "tobytes"):
        return content.tobytes()

    return bytes(content)


def normalize_uploaded_item(uploader_value):
    v = uploader_value

    if not v:
        return None

    if isinstance(v, (list, tuple)):
        return v[0]

    if isinstance(v, dict):
        return list(v.values())[0]

    raise TypeError(f"FileUpload.value の型が想定外: {type(v)}")


def save_uploaded_xlsx(uploader_value, save_as="uploaded_input.xlsx"):
    item = normalize_uploaded_item(uploader_value)

    if item is None:
        return None

    if isinstance(item, dict):
        content = item.get("content", None)
    else:
        try:
            content = item["content"]
        except Exception:
            content = getattr(item, "content", None)

    b = _bytes_from_uploaded_content(content)

    if b is None:
        raise KeyError("アップロードから content を取得できませんでした。")

    path = Path(save_as)

    with open(path, "wb") as f:
        f.write(b)

    return path


# ============================================================
# 4) 出力フォルダ
# ============================================================
def make_output_dirs(output_root=OUTPUT_ROOT, input_stem=None):
    """
    実行ごとに日時付きフォルダを作成する。

    例：
    LATEST
      └─ 20260703_133628_uploaded_input
          ├─ excel
          ├─ plots_png
          ├─ tables
          └─ logs
    """
    base_root = Path(output_root)

    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

    if input_stem:
        folder_name = f"{timestamp}_{safe_name(input_stem)}"
    else:
        folder_name = timestamp

    root = base_root / folder_name

    excel_dir = root / "excel"
    plot_dir = root / "plots_png"
    table_dir = root / "tables"
    log_dir = root / "logs"

    for d in [root, excel_dir, plot_dir, table_dir, log_dir]:
        d.mkdir(parents=True, exist_ok=True)

    return {
        "root": root,
        "excel": excel_dir,
        "plots": plot_dir,
        "tables": table_dir,
        "logs": log_dir,
        "run_label": folder_name
    }


def save_table_csv(df, path):
    if df is None:
        df = pd.DataFrame()

    df = clean_df_for_excel(df)

    df.to_csv(
        path,
        index=False,
        encoding="utf-8-sig"
    )


# ============================================================
# 5) 対象値範囲フィルタ
# ============================================================
def apply_value_range_filter(
    df,
    xcol,
    ycol,
    use_range=False,
    lo=None,
    hi=None,
    mode="both"
):
    if not use_range:
        return df.copy()

    if lo is None or hi is None:
        return df.copy()

    if lo > hi:
        lo, hi = hi, lo

    x = pd.to_numeric(df[xcol], errors="coerce")
    y = pd.to_numeric(df[ycol], errors="coerce")

    mx = pd.Series(True, index=df.index)
    my = pd.Series(True, index=df.index)

    mx &= x >= lo
    mx &= x <= hi
    my &= y >= lo
    my &= y <= hi

    if mode == "both":
        m = mx & my
    elif mode == "either":
        m = mx | my
    elif mode == "x_only":
        m = mx
    elif mode == "y_only":
        m = my
    else:
        m = mx & my

    return df.loc[m].copy()


# ============================================================
# 6) 統計・回帰
# ============================================================
def pearson_r(x, y):
    x = np.asarray(x, float)
    y = np.asarray(y, float)

    m = np.isfinite(x) & np.isfinite(y)
    x, y = x[m], y[m]

    if len(x) < 2:
        return np.nan

    return float(np.corrcoef(x, y)[0, 1])


def regression_fit(
    x,
    y,
    method="OLS",
    deming_lambda=1.0,
    theilsen_max_pairs=40000,
    seed=0
):
    x = np.asarray(x, float)
    y = np.asarray(y, float)

    m = np.isfinite(x) & np.isfinite(y)
    x, y = x[m], y[m]
    n = len(x)

    if n < 2:
        return np.nan, np.nan

    if method == "OLS":
        a, b = np.polyfit(x, y, 1)
        return float(a), float(b)

    if method == "Deming":
        lx = float(deming_lambda) if deming_lambda else 1.0

        xbar = x.mean()
        ybar = y.mean()

        sxx = np.mean((x - xbar) ** 2)
        syy = np.mean((y - ybar) ** 2)
        sxy = np.mean((x - xbar) * (y - ybar))

        if sxy == 0:
            a = 0.0
        else:
            a = (
                syy - lx * sxx
                + math.sqrt((syy - lx * sxx) ** 2 + 4 * lx * sxy * sxy)
            ) / (2 * sxy)

        b = ybar - a * xbar
        return float(a), float(b)

    if method == "TheilSen":
        rng = np.random.default_rng(seed)
        idx = np.arange(n)
        total_pairs = n * (n - 1) // 2
        slopes = []

        if total_pairs <= theilsen_max_pairs:
            for i in range(n - 1):
                dx = x[i + 1:] - x[i]
                dy = y[i + 1:] - y[i]
                mask = dx != 0
                slopes.extend((dy[mask] / dx[mask]).tolist())
        else:
            for _ in range(theilsen_max_pairs):
                i, j = rng.choice(idx, 2, replace=False)
                dx = x[j] - x[i]

                if dx == 0:
                    continue

                slopes.append((y[j] - y[i]) / dx)

        a = float(np.median(slopes)) if slopes else 0.0
        b = float(np.median(y - a * x))

        return a, b

    a, b = np.polyfit(x, y, 1)
    return float(a), float(b)


def passing_bablok_fit_with_ci(
    x,
    y,
    alpha=0.05,
    max_pairs=200000,
    seed=0
):
    x = np.asarray(x, float)
    y = np.asarray(y, float)

    m = np.isfinite(x) & np.isfinite(y)
    x, y = x[m], y[m]
    n = len(x)

    empty_info = {
        "slope_ci_low": np.nan,
        "slope_ci_high": np.nan,
        "intercept_ci_low": np.nan,
        "intercept_ci_high": np.nan,
        "pb_n_slopes": 0,
        "pb_ci_method": ""
    }

    if n < 3:
        empty_info["pb_ci_method"] = "insufficient n"
        return np.nan, np.nan, empty_info

    rng = np.random.default_rng(seed)
    total_pairs = n * (n - 1) // 2
    slopes = []

    if total_pairs <= max_pairs:
        for i in range(n - 1):
            dx = x[i + 1:] - x[i]
            dy = y[i + 1:] - y[i]

            mask = dx != 0
            s = dy[mask] / dx[mask]
            s = s[np.isfinite(s)]
            s = s[s != -1]

            slopes.extend(s.tolist())

        sampled = False
    else:
        idx = np.arange(n)

        for _ in range(max_pairs):
            i, j = rng.choice(idx, 2, replace=False)
            dx = x[j] - x[i]

            if dx == 0:
                continue

            s = (y[j] - y[i]) / dx

            if np.isfinite(s) and s != -1:
                slopes.append(float(s))

        sampled = True

    if len(slopes) == 0:
        empty_info["pb_ci_method"] = "no valid slopes"
        return np.nan, np.nan, empty_info

    slopes = np.sort(np.asarray(slopes, dtype=float))
    N = len(slopes)
    K = int(np.sum(slopes < -1))

    if N % 2 == 1:
        idx_med = (N - 1) // 2 + K
        idx_med = max(0, min(idx_med, N - 1))
        slope = float(slopes[idx_med])
    else:
        idx1 = N // 2 - 1 + K
        idx2 = N // 2 + K

        idx1 = max(0, min(idx1, N - 1))
        idx2 = max(0, min(idx2, N - 1))

        slope = float((slopes[idx1] + slopes[idx2]) / 2)

    intercept = float(np.median(y - slope * x))

    z = 1.959963984540054
    C = z * math.sqrt(n * (n - 1) * (2 * n + 5) / 18)

    m1 = int(math.floor((N - C) / 2 + K))
    m2 = int(math.ceil((N + C) / 2 + K))

    m1 = max(0, min(m1, N - 1))
    m2 = max(0, min(m2, N - 1))

    slope_low = float(slopes[min(m1, m2)])
    slope_high = float(slopes[max(m1, m2)])

    int_candidates = [
        np.median(y - slope_low * x),
        np.median(y - slope_high * x)
    ]

    intercept_low = float(np.nanmin(int_candidates))
    intercept_high = float(np.nanmax(int_candidates))

    info = {
        "slope_ci_low": slope_low,
        "slope_ci_high": slope_high,
        "intercept_ci_low": intercept_low,
        "intercept_ci_high": intercept_high,
        "pb_n_slopes": int(N),
        "pb_ci_method": "approx_rank_based_sampled" if sampled else "approx_rank_based_all_pairs"
    }

    return slope, intercept, info


def regression_fit_info(
    x,
    y,
    method="OLS",
    deming_lambda=1.0,
    theilsen_max_pairs=40000,
    seed=0
):
    info = {
        "slope_ci_low": np.nan,
        "slope_ci_high": np.nan,
        "intercept_ci_low": np.nan,
        "intercept_ci_high": np.nan,
        "pb_n_slopes": np.nan,
        "pb_ci_method": ""
    }

    if method == "PassingBablok":
        a, b, pb_info = passing_bablok_fit_with_ci(
            x,
            y,
            alpha=0.05,
            max_pairs=200000,
            seed=seed
        )
        info.update(pb_info)
        return a, b, info

    a, b = regression_fit(
        x,
        y,
        method=method,
        deming_lambda=deming_lambda,
        theilsen_max_pairs=theilsen_max_pairs,
        seed=seed
    )

    return a, b, info


# ============================================================
# 7) 乖離候補・検体別指標
# ============================================================
def mad(arr):
    arr = np.asarray(arr, float)
    med = np.nanmedian(arr)
    return np.nanmedian(np.abs(arr - med))


def compute_residuals(x, y, a, b):
    x = np.asarray(x, float)
    y = np.asarray(y, float)
    return y - (a * x + b)


def classify_outlier_level(abs_z):
    if not np.isfinite(abs_z):
        return "not_evaluable"

    if abs_z >= 5.0:
        return "strong_candidate"

    if abs_z >= 3.5:
        return "candidate"

    if abs_z >= 2.5:
        return "mild_candidate"

    return "none"


def compute_pair_sample_metrics(df, id_col, group_col, xcol, ycol, a, b):
    has_group = group_col and group_col in df.columns

    cols = [id_col, xcol, ycol] + ([group_col] if has_group else [])

    sub = df[cols].dropna(subset=[xcol, ycol]).copy()

    if sub.empty:
        return sub

    x = pd.to_numeric(sub[xcol], errors="coerce").astype(float)
    y = pd.to_numeric(sub[ycol], errors="coerce").astype(float)

    sub["X"] = xcol
    sub["Y"] = ycol

    sub["diff_y_minus_x"] = y - x
    sub["abs_diff_yx"] = np.abs(sub["diff_y_minus_x"])
    sub["mean_xy"] = (x + y) / 2.0

    sub["rel_diff_yx_pct"] = np.where(
        sub["mean_xy"] != 0,
        sub["abs_diff_yx"] / np.abs(sub["mean_xy"]) * 100,
        np.nan
    )

    sub["bias_pct_vs_x"] = np.where(
        x != 0,
        (y - x) / np.abs(x) * 100,
        np.nan
    )

    sub["ratio_y_over_x"] = np.where(
        x != 0,
        y / x,
        np.nan
    )

    sub["pred_y"] = a * x + b
    sub["residual"] = y - sub["pred_y"]
    sub["abs_residual"] = np.abs(sub["residual"])

    resid = sub["residual"].to_numpy(dtype=float)
    s = mad(resid)
    scale = 1.4826 * s if (np.isfinite(s) and s > 0) else np.nan

    if np.isfinite(scale):
        sub["z_MAD"] = sub["residual"] / scale
    else:
        sub["z_MAD"] = np.nan

    sub["abs_z_MAD"] = np.abs(sub["z_MAD"])
    sub["outlier_level"] = sub["abs_z_MAD"].apply(classify_outlier_level)

    sub["outlier_basis"] = "回帰残差 y-(slope*x+intercept) をMADで標準化"
    sub["outlier_note"] = "乖離候補であり、自動除外ではない"

    return sub


def pick_outliers_table(
    df,
    id_col,
    group_col,
    xcol,
    ycol,
    a,
    b,
    z_thresh=3.5,
    top_n=200
):
    sub = compute_pair_sample_metrics(
        df=df,
        id_col=id_col,
        group_col=group_col,
        xcol=xcol,
        ycol=ycol,
        a=a,
        b=b
    )

    if sub.empty or "abs_z_MAD" not in sub.columns:
        return sub, sub

    flagged = sub[
        np.isfinite(sub["abs_z_MAD"])
        & (sub["abs_z_MAD"] >= z_thresh)
    ].copy()

    flagged = flagged.sort_values("abs_z_MAD", ascending=False).head(top_n)

    return sub, flagged


# ============================================================
# 8) プロット作成
# ============================================================
def plot_suite(
    df,
    id_col,
    group_col,
    xcol,
    ycol,
    method,
    lam,
    a,
    b,
    r,
    fit_info=None,
    z_thresh=3.5,
    show_outlier_labels=True,
    outlier_label_top=8,
    add_diag=True,
    show_fit=True,
    show_stats=True,
    show_equation=True,
    show_ci=True,
    show_ba_text=True,
    show_outlier_text=True,
    normal_s=16,
    outlier_s=28,
    outlier_lw=1.2,
    fig_width=16,
    fig_height=10,
    dpi=150
):
    if fit_info is None:
        fit_info = {}

    has_group = group_col and group_col in df.columns

    base_cols = [id_col, xcol, ycol] + ([group_col] if has_group else [])

    sub = df[base_cols].dropna(subset=[xcol, ycol]).copy()

    if sub.empty:
        return None, None, None, None, None

    all_rows, flagged = pick_outliers_table(
        df=df,
        id_col=id_col,
        group_col=group_col,
        xcol=xcol,
        ycol=ycol,
        a=a,
        b=b,
        z_thresh=z_thresh,
        top_n=200
    )

    x = pd.to_numeric(sub[xcol], errors="coerce").astype(float).to_numpy()
    y = pd.to_numeric(sub[ycol], errors="coerce").astype(float).to_numpy()

    resid = compute_residuals(x, y, a, b)
    mean_xy = (x + y) / 2.0
    diff_yx = y - x

    bias = float(np.nanmean(diff_yx))
    sd = float(np.nanstd(diff_yx, ddof=1)) if len(diff_yx) > 1 else np.nan

    loa_hi = bias + 1.96 * sd if np.isfinite(sd) else np.nan
    loa_lo = bias - 1.96 * sd if np.isfinite(sd) else np.nan

    flagged_ids = set(flagged[id_col].tolist()) if flagged is not None and not flagged.empty else set()
    is_flagged = sub[id_col].isin(flagged_ids).to_numpy()

    if has_group:
        groups = sorted(sub[group_col].dropna().unique().tolist(), key=lambda t: str(t))
        cmap = plt.get_cmap("tab10")
        color_map = {g: cmap(i % 10) for i, g in enumerate(groups)}
        colors = sub[group_col].map(lambda g: color_map.get(g, "gray")).to_numpy()
    else:
        groups = []
        color_map = {}
        colors = np.array(["C0"] * len(sub))

    fig = plt.figure(figsize=(fig_width, fig_height), dpi=dpi)
    gs = fig.add_gridspec(2, 2)

    # ---------------- Scatter ----------------
    ax1 = fig.add_subplot(gs[0, 0])

    ax1.scatter(
        x[~is_flagged],
        y[~is_flagged],
        s=normal_s,
        alpha=0.75,
        c=colors[~is_flagged]
    )

    if np.any(is_flagged):
        ax1.scatter(
            x[is_flagged],
            y[is_flagged],
            s=outlier_s,
            alpha=0.95,
            facecolors="none",
            edgecolors="red",
            linewidths=outlier_lw
        )

    ax1.set_title(f"散布図：{ycol} vs {xcol} / {REGRESSION_LABELS.get(method, method)}")
    ax1.set_xlabel(xcol)
    ax1.set_ylabel(ycol)
    ax1.grid(True, alpha=0.25)

    lo_xy = float(np.nanmin([np.nanmin(x), np.nanmin(y)]))
    hi_xy = float(np.nanmax([np.nanmax(x), np.nanmax(y)]))

    if add_diag:
        ax1.plot(
            [lo_xy, hi_xy],
            [lo_xy, hi_xy],
            "--",
            lw=1,
            alpha=0.6,
            label="y=x"
        )

    if show_fit and np.isfinite(a) and np.isfinite(b):
        xx = np.array([lo_xy, hi_xy])
        ax1.plot(
            xx,
            a * xx + b,
            lw=1.8,
            alpha=0.85,
            label="回帰直線"
        )

    if show_stats:
        lines = []
        lines.append(f"n={len(x)}")
        lines.append(f"Pearson r={r:.4f}")
        lines.append(
            REGRESSION_LABELS.get(method, method)
            + (f" (λ={lam:g})" if method == "Deming" else "")
        )

        if show_equation:
            lines.append(f"y={a:.4f}x+{b:.4f}")

        if show_ci:
            sl = fit_info.get("slope_ci_low", np.nan)
            sh = fit_info.get("slope_ci_high", np.nan)
            il = fit_info.get("intercept_ci_low", np.nan)
            ih = fit_info.get("intercept_ci_high", np.nan)

            if np.isfinite(sl) and np.isfinite(sh):
                lines.append(f"slope 95%CI [{sl:.4f}, {sh:.4f}]")

            if np.isfinite(il) and np.isfinite(ih):
                lines.append(f"intercept 95%CI [{il:.4f}, {ih:.4f}]")

        if show_ba_text:
            lines.append(f"BA bias={bias:.4g}")

            if np.isfinite(loa_lo) and np.isfinite(loa_hi):
                lines.append(f"LoA [{loa_lo:.4g}, {loa_hi:.4g}]")

        if show_outlier_text:
            n_flag = int(len(flagged)) if flagged is not None else 0
            lines.append(f"乖離候補: |z_MAD|≥{z_thresh}")
            lines.append(f"乖離候補数={n_flag}")

        ax1.text(
            0.03,
            0.97,
            "\n".join(lines),
            transform=ax1.transAxes,
            va="top",
            fontsize=9,
            bbox=dict(boxstyle="round", facecolor="white", alpha=0.75)
        )

    # ---------------- 乖離IDラベル ----------------
    if (
        show_outlier_labels
        and np.any(is_flagged)
        and outlier_label_top > 0
        and flagged is not None
        and not flagged.empty
    ):
        top_flagged = flagged.sort_values("abs_z_MAD", ascending=False).head(outlier_label_top)

        for _, row in top_flagged.iterrows():
            try:
                xx0 = float(row[xcol])
                yy0 = float(row[ycol])
                sid = row[id_col]

                if np.isfinite(xx0) and np.isfinite(yy0):
                    ax1.text(
                        xx0,
                        yy0,
                        str(sid),
                        fontsize=8,
                        color="red"
                    )
            except Exception:
                continue

    if has_group:
        for g in groups[:10]:
            ax1.scatter([], [], c=[color_map[g]], label=f"{group_col}:{g}")

    ax1.legend(fontsize=8, loc="best")

    # ---------------- Bland-Altman ----------------
    ax2 = fig.add_subplot(gs[0, 1])

    ax2.scatter(
        mean_xy[~is_flagged],
        diff_yx[~is_flagged],
        s=normal_s,
        alpha=0.75,
        c=colors[~is_flagged]
    )

    if np.any(is_flagged):
        ax2.scatter(
            mean_xy[is_flagged],
            diff_yx[is_flagged],
            s=outlier_s,
            alpha=0.95,
            facecolors="none",
            edgecolors="red",
            linewidths=outlier_lw
        )

    ax2.axhline(
        bias,
        color="black",
        lw=1.5,
        label=f"平均差={bias:.3g}"
    )

    if np.isfinite(loa_hi) and np.isfinite(loa_lo):
        ax2.axhline(
            loa_hi,
            color="gray",
            lw=1.2,
            ls="--",
            label=f"+1.96SD={loa_hi:.3g}"
        )

        ax2.axhline(
            loa_lo,
            color="gray",
            lw=1.2,
            ls="--",
            label=f"-1.96SD={loa_lo:.3g}"
        )

    ax2.set_title("Bland–Altman：差(Y-X) vs 平均((X+Y)/2)")
    ax2.set_xlabel("平均 (X+Y)/2")
    ax2.set_ylabel("差 (Y-X)")
    ax2.grid(True, alpha=0.25)
    ax2.legend(fontsize=8, loc="best")

    # ---------------- Residuals ----------------
    ax3 = fig.add_subplot(gs[1, 0])

    ax3.scatter(
        x[~is_flagged],
        resid[~is_flagged],
        s=normal_s,
        alpha=0.75,
        c=colors[~is_flagged]
    )

    if np.any(is_flagged):
        ax3.scatter(
            x[is_flagged],
            resid[is_flagged],
            s=outlier_s,
            alpha=0.95,
            facecolors="none",
            edgecolors="red",
            linewidths=outlier_lw
        )

    ax3.axhline(0, color="black", lw=1.2)
    ax3.set_title("残差プロット：残差(Y-(aX+b)) vs X")
    ax3.set_xlabel(xcol)
    ax3.set_ylabel("残差")
    ax3.grid(True, alpha=0.25)

    # ---------------- Text panel ----------------
    ax4 = fig.add_subplot(gs[1, 1])
    ax4.axis("off")

    n_flagged = int(len(flagged)) if flagged is not None else 0

    txt = (
        f"【統計まとめ】\n"
        f"X={xcol}\n"
        f"Y={ycol}\n"
        f"n={len(x)}\n"
        f"Pearson r={r:.6f}\n"
        f"回帰: {REGRESSION_LABELS.get(method, method)}"
        + (f" (λ={lam:g})" if method == "Deming" else "")
        + "\n"
        f"slope={a:.6f}\n"
        f"intercept={b:.6f}\n"
    )

    sl = fit_info.get("slope_ci_low", np.nan)
    sh = fit_info.get("slope_ci_high", np.nan)
    il = fit_info.get("intercept_ci_low", np.nan)
    ih = fit_info.get("intercept_ci_high", np.nan)

    if show_ci:
        if np.isfinite(sl) and np.isfinite(sh):
            txt += f"slope 95%CI=[{sl:.6g}, {sh:.6g}]\n"

        if np.isfinite(il) and np.isfinite(ih):
            txt += f"intercept 95%CI=[{il:.6g}, {ih:.6g}]\n"

    txt += (
        f"\n【Bland–Altman】\n"
        f"bias(Y-X)={bias:.6g}\n"
    )

    if np.isfinite(sd):
        txt += f"SD={sd:.6g}\nLoA=[{loa_lo:.6g}, {loa_hi:.6g}]\n"

    txt += (
        f"\n【乖離候補】\n"
        f"|z_MAD|≥{z_thresh}\n"
        f"乖離候補数={n_flagged}\n"
        f"※乖離候補は自動除外していません\n"
        f"※r/slope/interceptは解析対象全データで算出\n"
    )

    ax4.text(
        0.02,
        0.98,
        txt,
        va="top",
        fontsize=10
    )

    fig.tight_layout()

    return fig, sub, flagged, bias, (loa_lo, loa_hi)


# ============================================================
# 9) Excel出力
# ============================================================
def _excel_safe_value(v):
    if isinstance(v, (np.integer,)):
        return int(v)

    if isinstance(v, (np.floating,)):
        if np.isnan(v) or np.isinf(v):
            return None
        return float(v)

    if isinstance(v, float):
        if math.isnan(v) or math.isinf(v):
            return None
        return v

    if pd.isna(v):
        return None

    return v


def clean_df_for_excel(df):
    if df is None:
        return pd.DataFrame()

    out = df.copy()
    out = out.replace([np.inf, -np.inf], np.nan)
    out = out.where(pd.notna(out), None)

    return out


def insert_images_into_excel(
    input_xlsx,
    output_xlsx,
    image_paths,
    plot_sheet=SHEET_PLOTS,
    start_cell="A1",
    row_step=44,
    max_width_px=1100
):
    wb = load_workbook(input_xlsx)

    if plot_sheet in wb.sheetnames:
        wb.remove(wb[plot_sheet])

    ws = wb.create_sheet(plot_sheet)

    m = re.match(r"([A-Z]+)(\d+)", start_cell)

    if not m:
        start_cell = "A1"
        m = re.match(r"([A-Z]+)(\d+)", start_cell)

    col_letters = m.group(1)
    row = int(m.group(2))

    for p in image_paths:
        img = XLImage(str(p))

        if img.width and img.width > max_width_px:
            scale = max_width_px / img.width
            img.width = int(img.width * scale)
            img.height = int(img.height * scale)

        ws.add_image(img, f"{col_letters}{row}")
        row += row_step

    wb.save(output_xlsx)


def write_df_to_sheet(xlsx_path, df, sheet_name, index=False):
    df = clean_df_for_excel(df)

    wb = load_workbook(xlsx_path)

    if sheet_name in wb.sheetnames:
        wb.remove(wb[sheet_name])

    ws = wb.create_sheet(sheet_name)

    df_to_write = df.reset_index() if index else df

    ws.append(list(df_to_write.columns))

    for row in df_to_write.itertuples(index=False):
        ws.append([_excel_safe_value(v) for v in row])

    wb.save(xlsx_path)


# ============================================================
# 10) 種別ごとの統計
# ============================================================
def groupwise_stats(df, id_col, group_col, xcol, ycol, method, lam):
    if not (group_col and group_col in df.columns):
        return pd.DataFrame(columns=[
            "X",
            "Y",
            "group",
            "n",
            "r",
            "slope",
            "intercept",
            "regression",
            "lambda"
        ])

    rows = []

    sub = df[[group_col, xcol, ycol]].dropna(subset=[xcol, ycol]).copy()

    for g, gdf in sub.groupby(group_col, dropna=False):
        x = pd.to_numeric(gdf[xcol], errors="coerce").astype(float).to_numpy()
        y = pd.to_numeric(gdf[ycol], errors="coerce").astype(float).to_numpy()

        if len(x) < 2:
            continue

        r = pearson_r(x, y)
        a, b, fit_info = regression_fit_info(
            x,
            y,
            method=method,
            deming_lambda=lam
        )

        rows.append({
            "X": xcol,
            "Y": ycol,
            "group": str(g),
            "n": len(x),
            "r": r,
            "slope": a,
            "intercept": b,
            "regression": method,
            "regression_label": REGRESSION_LABELS.get(method, method),
            "lambda": lam if method == "Deming" else np.nan,
            "slope_95CI_low": fit_info.get("slope_ci_low", np.nan),
            "slope_95CI_high": fit_info.get("slope_ci_high", np.nan),
            "intercept_95CI_low": fit_info.get("intercept_ci_low", np.nan),
            "intercept_95CI_high": fit_info.get("intercept_ci_high", np.nan),
            "outlier_excluded_from_fit": False,
            "fit_data_note": "群別統計も乖離候補を除外せず算出"
        })

    return pd.DataFrame(rows)


# ============================================================
# 11) 指標説明シート
# ============================================================
def make_metric_definitions_df():
    rows = [
        ["入力データ", "検体ID / SampleID / ID", "各検体を識別する列", "入力列", "図中IDラベル、outliers、sample_metricsで検体を追跡", "列名がない場合は先頭列をID列として扱います"],
        ["入力データ", "種別 / Group / Type", "任意の分類列", "入力列", "存在し、分類列と判断できる場合のみ色分け・群別統計に使用", "なくても動きます"],
        ["入力データ", "処方列", "比較対象となる数値列", "入力列", "列名が「処方」で始まる列を優先。なければ数値列を候補", "2列以上必要です"],
        ["基本情報", "X", "比較ペアにおける基準側の列名", "列名", "例：X=処方1", ""],
        ["基本情報", "Y", "比較ペアにおける比較側の列名", "列名", "例：Y=処方2", ""],
        ["差分", "diff_y_minus_x", "YとXの単純差", "Y - X", "正ならYがXより高く、負なら低い", "Bland–Altmanの差と同じです"],
        ["差分", "abs_diff_yx", "YとXの差の絶対値", "|Y - X|", "方向を問わず離れ具合を見る", "単位依存です"],
        ["Bland–Altman", "mean_xy", "XとYの平均値", "(X + Y) / 2", "BAプロットの横軸", ""],
        ["相対差", "rel_diff_yx_pct", "平均値基準の相対差", "|Y - X| / |(X+Y)/2| × 100", "値域差をならしたズレ", "平均が0に近いと不安定"],
        ["相対差", "bias_pct_vs_x", "X基準の差の割合", "(Y - X) / |X| × 100", "Xに対してYが何%高い/低いか", "Xが0に近いと不安定"],
        ["比率", "ratio_y_over_x", "Y/X比", "Y / X", "1に近いほど近い", "Xが0に近いと不安定"],
        ["回帰", "pred_y", "回帰式から予測されるY", "slope × X + intercept", "全体傾向から期待されるY", ""],
        ["回帰残差", "residual", "実測Yと予測Yの差", "Y - pred_y", "回帰直線からのズレ", "単純なY-Xではありません"],
        ["回帰残差", "abs_residual", "残差の絶対値", "|Y - pred_y|", "方向を問わず回帰からの外れ具合", "単位依存です"],
        ["乖離候補", "z_MAD", "残差をMADで標準化した指標", "residual / (1.4826 × MAD(residual))", "全体の残差ばらつきに対する外れ具合", "自動除外基準ではありません"],
        ["乖離候補", "abs_z_MAD", "z_MADの絶対値", "|z_MAD|", "大きいほど乖離候補", "方向はz_MADまたはresidualで確認"],
        ["乖離候補", "outlier_level", "確認優先度", "2.5以上=mild, 3.5以上=candidate, 5.0以上=strong", "乖離候補の段階表示", "除外判定ではありません"],
        ["回帰統計", "r", "Pearson相関係数", "corr(X, Y)", "1に近いほど正の直線相関が強い", "一致性ではありません"],
        ["回帰統計", "slope", "回帰直線の傾き", "回帰法に依存", "1に近いほど比例性が近い", "回帰法により変わります"],
        ["回帰統計", "intercept", "回帰直線の切片", "回帰法に依存", "0に近いほど定数差が小さい", "測定範囲外の解釈注意"],
        ["回帰統計", "slope_95CI_low/high", "傾きの95%信頼区間", "Passing-Bablokで算出", "傾き推定の不確かさ", "現コードでは主にPassing-Bablokのみ"],
        ["回帰統計", "intercept_95CI_low/high", "切片の95%信頼区間", "Passing-Bablokで算出", "切片推定の不確かさ", "現コードでは主にPassing-Bablokのみ"],
        ["Bland–Altman", "BA_bias(y-x)", "Y-Xの平均差", "mean(Y - X)", "平均的な差", "外れ値の影響を受けます"],
        ["Bland–Altman", "BA_LoA_low / BA_LoA_high", "95%一致限界", "bias ± 1.96×SD(Y-X)", "差が多く入る想定範囲", "差の分布に依存"],
        ["仕様", "outlier_excluded_from_fit", "乖離候補を回帰計算から除外したか", "False=除外していない", "r/slope/interceptは解析対象全データで算出", "対象範囲フィルタで外れた値は除外"]
    ]

    return pd.DataFrame(
        rows,
        columns=["分類", "指標名", "意味", "計算式", "解釈", "注意点"]
    )


# ============================================================
# 12) UI
# ============================================================
help_html = widgets.HTML("""
<div style="border:1px solid #ccc; padding:12px; border-radius:8px; background:#fafafa; line-height:1.6;">
<b>このツールの目的</b><br>
処方列どうしの相関・回帰・Bland–Altman・残差・乖離候補をまとめて確認し、Excelに出力します。<br>
乖離候補は自動除外ではなく、確認すべき候補として出力します。<br><br>

<b>必要な入力データ</b><br>
<ul>
<li><b>検体ID列</b>：推奨列名は「検体ID」「SampleID」「ID」です。ない場合は先頭列をIDとして扱います。</li>
<li><b>処方列</b>：比較したい数値列です。列名が「処方」で始まる列を優先して検出します。</li>
<li><b>種別列</b>：任意です。「種別」「Group」「Type」があれば群別色分け・群別統計に使います。なくても動きます。</li>
</ul>

<b>主な設定値</b><br>
<ul>
<li><b>全回帰法で出力</b>：ONにすると OLS / Deming / Theil-Sen / Passing-Bablok の全てで図・統計を出力します。</li>
<li><b>回帰法</b>：全回帰法OFFのときに使用する回帰法です。</li>
<li><b>ベスト</b>：初期設定は r最大（正相関優先）です。</li>
<li><b>対象範囲</b>：下限・上限に数値を入力します。通常はOFFで良いです。</li>
<li><b>乖離z(MAD)</b>：回帰残差をMADで標準化した値です。乖離候補の抽出に使います。</li>
<li><b>乖離IDラベル表示</b>：ONにすると乖離候補の検体IDを散布図上に表示します。</li>
<li><b>95%CI表示</b>：グラフ内テキストに95%CIを表示します。CIの線や帯は描画しません。</li>
</ul>

<b>重要</b><br>
相関係数・傾き・切片は、乖離候補を除外せずに算出します。<br>
除外するかどうかは、測定情報・入力情報・検体情報を確認してから判断してください。
</div>
""")

uploader = widgets.FileUpload(
    accept=".xlsx",
    multiple=False,
    description="Excelをアップロード"
)

sheet_dd = widgets.Dropdown(
    options=["(未選択)"],
    value="(未選択)",
    description="Sheet"
)

mode_dd = widgets.Dropdown(
    options=[
        ("全組合せ", "all"),
        ("隣同士のみ", "adjacent"),
        ("基準処方 vs その他", "baseline")
    ],
    value="baseline",
    description="モード"
)

baseline_sel = widgets.SelectMultiple(
    options=[],
    value=(),
    description="基準処方",
    rows=8
)

all_reg_ck = widgets.Checkbox(
    value=False,
    description="全回帰法で出力"
)

reg_dd = widgets.Dropdown(
    options=[
        ("OLS（最小二乗）", "OLS"),
        ("Deming（両軸誤差）", "Deming"),
        ("Theil-Sen（ロバスト）", "TheilSen"),
        ("Passing-Bablok（ノンパラメトリック）", "PassingBablok"),
    ],
    value="PassingBablok",
    description="回帰法"
)

deming_lambda = widgets.FloatText(
    value=1.0,
    description="λ(Deming)"
)

best_mode = widgets.ToggleButtons(
    options=[
        ("|r|最大", "abs"),
        ("r最大（正相関優先）", "pos")
    ],
    value="pos",
    description="ベスト"
)

z_thresh = widgets.FloatSlider(
    value=3.5,
    min=1.5,
    max=8.0,
    step=0.1,
    description="乖離z(MAD)"
)

show_outlier_label_ck = widgets.Checkbox(
    value=True,
    description="乖離IDラベル表示"
)

label_top = widgets.IntSlider(
    value=8,
    min=0,
    max=30,
    step=1,
    description="乖離IDラベル数"
)

use_range_ck = widgets.Checkbox(
    value=False,
    description="対象範囲で絞る"
)

range_min_txt = widgets.FloatText(
    value=0.0,
    description="下限"
)

range_max_txt = widgets.FloatText(
    value=100.0,
    description="上限"
)

range_mode_dd = widgets.Dropdown(
    options=[
        ("XとYの両方が範囲内", "both"),
        ("XまたはYが範囲内", "either"),
        ("Xのみ範囲内", "x_only"),
        ("Yのみ範囲内", "y_only"),
    ],
    value="both",
    description="範囲条件"
)

add_diag_ck = widgets.Checkbox(
    value=True,
    description="y=x補助線"
)

show_fit_ck = widgets.Checkbox(
    value=True,
    description="回帰直線"
)

show_stats_ck = widgets.Checkbox(
    value=True,
    description="統計表示"
)

show_py_ck = widgets.Checkbox(
    value=True,
    description="Python上に表示"
)

max_show = widgets.IntSlider(
    value=6,
    min=0,
    max=30,
    step=1,
    description="表示上限"
)

show_equation_ck = widgets.Checkbox(
    value=True,
    description="回帰式を表示"
)

show_ci_ck = widgets.Checkbox(
    value=True,
    description="95%CIを表示"
)

show_ba_text_ck = widgets.Checkbox(
    value=True,
    description="BA統計を表示"
)

show_outlier_text_ck = widgets.Checkbox(
    value=True,
    description="乖離数を表示"
)

normal_point_size = widgets.IntSlider(
    value=16,
    min=4,
    max=80,
    step=2,
    description="通常点サイズ"
)

outlier_point_size = widgets.IntSlider(
    value=28,
    min=6,
    max=120,
    step=2,
    description="乖離○サイズ"
)

outlier_line_width = widgets.FloatSlider(
    value=1.2,
    min=0.5,
    max=4.0,
    step=0.1,
    description="乖離○線幅"
)

fig_width_slider = widgets.IntSlider(
    value=16,
    min=8,
    max=28,
    step=1,
    description="図横サイズ"
)

fig_height_slider = widgets.IntSlider(
    value=10,
    min=6,
    max=20,
    step=1,
    description="図縦サイズ"
)

plot_dpi_slider = widgets.IntSlider(
    value=150,
    min=80,
    max=300,
    step=10,
    description="DPI"
)

load_btn = widgets.Button(
    description="列を読み込み（基準候補更新）",
    button_style="info"
)

run_btn = widgets.Button(
    description="実行（解析→Excel出力）",
    button_style="success"
)

status = widgets.Output()
plots_out = widgets.Output()

display(widgets.VBox([
    widgets.HTML("<b>①アップロード → ②列読み込み → ③実行</b>"),
    help_html,
    uploader,
    sheet_dd,
    widgets.HBox([mode_dd, all_reg_ck, reg_dd, deming_lambda]),
    widgets.HBox([baseline_sel, widgets.VBox([best_mode, z_thresh, show_outlier_label_ck, label_top])]),
    widgets.HBox([use_range_ck, range_min_txt, range_max_txt, range_mode_dd]),
    widgets.HBox([add_diag_ck, show_fit_ck, show_stats_ck, show_py_ck, max_show]),
    widgets.HBox([show_equation_ck, show_ci_ck, show_ba_text_ck, show_outlier_text_ck]),
    widgets.HBox([normal_point_size, outlier_point_size, outlier_line_width]),
    widgets.HBox([fig_width_slider, fig_height_slider, plot_dpi_slider]),
    widgets.HBox([load_btn, run_btn]),
    status,
    plots_out
]))

_state = {
    "xlsx_path": None,
    "df": None,
    "id_col": None,
    "group_col": None,
    "value_cols": None
}


# ============================================================
# 13) UI callbacks
# ============================================================
def refresh_sheet_list(xlsx_path):
    xl = pd.ExcelFile(xlsx_path, engine="openpyxl")
    sheets = xl.sheet_names

    if sheets:
        sheet_dd.options = sheets
        sheet_dd.value = sheets[0]
    else:
        sheet_dd.options = ["(未選択)"]
        sheet_dd.value = "(未選択)"


def set_default_range_from_data(df, value_cols):
    vals = []

    for c in value_cols:
        s = pd.to_numeric(df[c], errors="coerce")
        vals.append(s)

    if not vals:
        return

    allv = pd.concat(vals, axis=0).dropna()

    if allv.empty:
        return

    lo0 = float(np.nanmin(allv))
    hi0 = float(np.nanmax(allv))

    if np.isfinite(lo0):
        range_min_txt.value = lo0

    if np.isfinite(hi0):
        range_max_txt.value = hi0


def on_upload_change(change):
    with status:
        clear_output()

        try:
            xlsx_path = save_uploaded_xlsx(
                uploader.value,
                save_as="uploaded_input.xlsx"
            )
        except Exception as e:
            print("アップロード読み込みエラー:", repr(e))
            return

        if xlsx_path is None:
            print("Excelをアップロードしてください。")
            return

        _state["xlsx_path"] = xlsx_path

        print("アップロードOK:", xlsx_path)

        refresh_sheet_list(xlsx_path)

        print("Sheet:", sheet_dd.value)


uploader.observe(on_upload_change, names="value")


def load_columns(_=None):
    with status:
        clear_output()

        xlsx_path = _state.get("xlsx_path")

        if xlsx_path is None:
            print("先にExcelをアップロードしてください。")
            return

        if sheet_dd.value == "(未選択)":
            print("Sheetを選択してください。")
            return

        df = pd.read_excel(
            xlsx_path,
            sheet_name=sheet_dd.value,
            engine="openpyxl"
        )

        id_col = pick_col(
            df,
            ID_COL_CANDIDATES,
            default=df.columns[0]
        )

        group_col_raw = pick_col(
            df,
            GROUP_COL_CANDIDATES,
            default=None
        )

        group_col = normalize_group_col(df, group_col_raw)

        value_cols = detect_value_cols(
            df,
            id_col,
            group_col,
            prefixes=VALUE_PREFIXES
        )

        _state.update({
            "df": df,
            "id_col": id_col,
            "group_col": group_col,
            "value_cols": value_cols
        })

        baseline_sel.options = value_cols

        if value_cols:
            baseline_sel.value = (value_cols[0],)

        set_default_range_from_data(df, value_cols)

        print("列読み込み完了")
        print("  ID列   :", id_col)
        print("  種別列 :", group_col if group_col else "(なし)")
        print("  比較列 :", value_cols)
        print(f"  対象範囲初期値: {range_min_txt.value:.4g} ～ {range_max_txt.value:.4g}")

        if len(value_cols) < 2:
            print("注意：比較列が2列未満です。解析には2列以上必要です。")


load_btn.on_click(load_columns)


def run_all(_=None):
    with status:
        clear_output()

    with plots_out:
        clear_output()

    xlsx_path = _state.get("xlsx_path")

    if xlsx_path is None:
        with status:
            print("先にExcelをアップロードしてください。")
        return

    if _state.get("df") is None:
        load_columns()

    df = _state.get("df")
    id_col = _state.get("id_col")
    group_col = normalize_group_col(df, _state.get("group_col"))
    value_cols = _state.get("value_cols")

    if df is None or id_col is None or not value_cols or len(value_cols) < 2:
        with status:
            print("解析に必要な列が不足しています。")
            print("検体ID列と、2列以上の比較列が必要です。")
        return

    if mode_dd.value == "adjacent":
        pairs = list(zip(value_cols[:-1], value_cols[1:]))

    elif mode_dd.value == "all":
        pairs = list(itertools.combinations(value_cols, 2))

    else:
        bases = list(baseline_sel.value) if baseline_sel.value else [value_cols[0]]
        pairs = []

        for b0 in bases:
            for c in value_cols:
                if c != b0:
                    pairs.append((b0, c))

    methods = REGRESSION_METHODS_ALL if all_reg_ck.value else [reg_dd.value]

    lam = deming_lambda.value

    lo_range = float(range_min_txt.value)
    hi_range = float(range_max_txt.value)

    if lo_range > hi_range:
        lo_range, hi_range = hi_range, lo_range

    try:
        dirs = make_output_dirs(
            OUTPUT_ROOT,
            input_stem=xlsx_path.stem
        )
    except Exception as e:
        with status:
            print("出力先フォルダ作成エラー:", repr(e))
        return

    out_dir = dirs["plots"]
    excel_dir = dirs["excel"]

    summary_rows = []
    group_rows = []
    outlier_tables = []
    sample_metric_tables = []
    img_paths = []
    shown = 0

    for method in methods:
        for xcol, ycol in pairs:
            df_pair = apply_value_range_filter(
                df,
                xcol=xcol,
                ycol=ycol,
                use_range=use_range_ck.value,
                lo=lo_range,
                hi=hi_range,
                mode=range_mode_dd.value
            )

            sub = df_pair[[xcol, ycol]].dropna()

            if len(sub) < 2:
                continue

            x = pd.to_numeric(sub[xcol], errors="coerce").astype(float).to_numpy()
            y = pd.to_numeric(sub[ycol], errors="coerce").astype(float).to_numpy()

            r = pearson_r(x, y)

            a, b, fit_info = regression_fit_info(
                x,
                y,
                method=method,
                deming_lambda=lam
            )

            fig, used_sub, flagged, bias, loa = plot_suite(
                df=df_pair,
                id_col=id_col,
                group_col=group_col,
                xcol=xcol,
                ycol=ycol,
                method=method,
                lam=lam,
                a=a,
                b=b,
                r=r,
                fit_info=fit_info,
                z_thresh=z_thresh.value,
                show_outlier_labels=show_outlier_label_ck.value,
                outlier_label_top=label_top.value,
                add_diag=add_diag_ck.value,
                show_fit=show_fit_ck.value,
                show_stats=show_stats_ck.value,
                show_equation=show_equation_ck.value,
                show_ci=show_ci_ck.value,
                show_ba_text=show_ba_text_ck.value,
                show_outlier_text=show_outlier_text_ck.value,
                normal_s=normal_point_size.value,
                outlier_s=outlier_point_size.value,
                outlier_lw=outlier_line_width.value,
                fig_width=fig_width_slider.value,
                fig_height=fig_height_slider.value,
                dpi=plot_dpi_slider.value
            )

            if fig is None:
                continue

            if show_py_ck.value and max_show.value > 0 and shown < max_show.value:
                with plots_out:
                    display(fig)

                shown += 1

            png = out_dir / f"QC_{safe_name(method)}_{safe_name(ycol)}_vs_{safe_name(xcol)}.png"

            fig.savefig(
                png,
                bbox_inches="tight"
            )

            plt.close(fig)

            img_paths.append(png)

            n_out = int(len(flagged)) if flagged is not None else 0

            summary_rows.append({
                "regression": method,
                "regression_label": REGRESSION_LABELS.get(method, method),
                "X": xcol,
                "Y": ycol,
                "n": len(x),
                "r": r,
                "|r|": abs(r) if np.isfinite(r) else np.nan,
                "slope": a,
                "intercept": b,
                "lambda": lam if method == "Deming" else np.nan,
                "slope_95CI_low": fit_info.get("slope_ci_low", np.nan),
                "slope_95CI_high": fit_info.get("slope_ci_high", np.nan),
                "intercept_95CI_low": fit_info.get("intercept_ci_low", np.nan),
                "intercept_95CI_high": fit_info.get("intercept_ci_high", np.nan),
                "PB_n_slopes": fit_info.get("pb_n_slopes", np.nan),
                "PB_CI_method": fit_info.get("pb_ci_method", ""),
                "BA_bias(y-x)": bias,
                "BA_LoA_low": loa[0],
                "BA_LoA_high": loa[1],
                "n_outliers": n_out,
                "outlier_z_thresh": z_thresh.value,
                "outlier_basis": "回帰残差 y-(slope*x+intercept) をMADで標準化",
                "outlier_note": "乖離候補であり、自動除外ではない",
                "outlier_excluded_from_fit": False,
                "fit_data_note": "相関係数・回帰係数は乖離候補を除外せず、解析対象データ全体で算出",
                "range_filter_used": bool(use_range_ck.value),
                "range_min": lo_range if use_range_ck.value else np.nan,
                "range_max": hi_range if use_range_ck.value else np.nan,
                "range_condition": range_mode_dd.value if use_range_ck.value else "",
                "show_outlier_labels": bool(show_outlier_label_ck.value),
                "outlier_label_top": label_top.value,
                "normal_point_size": normal_point_size.value,
                "outlier_point_size": outlier_point_size.value,
                "outlier_line_width": outlier_line_width.value,
                "figure_width": fig_width_slider.value,
                "figure_height": fig_height_slider.value,
                "plot_dpi": plot_dpi_slider.value,
                "output_root": str(dirs["root"])
            })

            gdf = groupwise_stats(
                df_pair,
                id_col,
                group_col,
                xcol,
                ycol,
                method,
                lam
            )

            if not gdf.empty:
                group_rows.append(gdf)

            if flagged is not None and not flagged.empty:
                outlier_tables.append(flagged.assign(
                    regression=method,
                    regression_label=REGRESSION_LABELS.get(method, method),
                    slope=a,
                    intercept=b,
                    r=r
                ))

            metrics_df = compute_pair_sample_metrics(
                df=df_pair,
                id_col=id_col,
                group_col=group_col,
                xcol=xcol,
                ycol=ycol,
                a=a,
                b=b
            )

            if not metrics_df.empty:
                metrics_df["regression"] = method
                metrics_df["regression_label"] = REGRESSION_LABELS.get(method, method)
                metrics_df["slope"] = a
                metrics_df["intercept"] = b
                metrics_df["r"] = r
                metrics_df["slope_95CI_low"] = fit_info.get("slope_ci_low", np.nan)
                metrics_df["slope_95CI_high"] = fit_info.get("slope_ci_high", np.nan)
                metrics_df["intercept_95CI_low"] = fit_info.get("intercept_ci_low", np.nan)
                metrics_df["intercept_95CI_high"] = fit_info.get("intercept_ci_high", np.nan)
                metrics_df["outlier_z_thresh"] = z_thresh.value
                metrics_df["outlier_excluded_from_fit"] = False
                metrics_df["fit_data_note"] = "相関係数・回帰係数は乖離候補を除外せず、解析対象データ全体で算出"
                metrics_df["range_filter_used"] = bool(use_range_ck.value)
                metrics_df["range_min"] = lo_range if use_range_ck.value else np.nan
                metrics_df["range_max"] = hi_range if use_range_ck.value else np.nan
                metrics_df["range_condition"] = range_mode_dd.value if use_range_ck.value else ""

                sample_metric_tables.append(metrics_df)

    if not summary_rows:
        with status:
            print("作図・解析できるペアがありませんでした。")
            print("欠損が多い、対象範囲が狭すぎる、または比較列が不足している可能性があります。")
        return

    summary_df = pd.DataFrame(summary_rows)

    if best_mode.value == "abs":
        best = summary_df.sort_values("|r|", ascending=False).iloc[0]
        best_label = "|r|最大"
    else:
        best = summary_df.sort_values("r", ascending=False).iloc[0]
        best_label = "r最大（正相関優先）"

    with status:
        print("=== 実行結果 ===")
        print("入力:", xlsx_path)
        print("出力root:", dirs["root"])
        print("モード:", mode_dd.value, " / ペア数:", len(pairs))
        print(
            "回帰法:",
            "全回帰法" if all_reg_ck.value else REGRESSION_LABELS.get(reg_dd.value, reg_dd.value)
        )
        print(
            f"ベスト相関（{best_label}）: "
            f"{best['regression_label']} / {best['Y']} vs {best['X']}"
        )
        print(
            f"  n={int(best['n'])}  "
            f"r={best['r']:.4f}  "
            f"slope={best['slope']:.4f}  "
            f"intercept={best['intercept']:.4f}"
        )
        print(
            f"  BA bias={best['BA_bias(y-x)']:.4g}  "
            f"LoA=[{best['BA_LoA_low']:.4g}, {best['BA_LoA_high']:.4g}]  "
            f"outliers={int(best['n_outliers'])}"
        )
        print("※相関係数・回帰係数は、乖離候補を除外せずに算出しています。")
        print("※95%CI表示は、グラフ内テキスト表示のON/OFFです。CI線・CI帯は描画していません。")
        print(f"Python表示枚数: {shown}")

    output_xlsx = excel_dir / f"{xlsx_path.stem}{OUT_SUFFIX}{xlsx_path.suffix}"

    insert_images_into_excel(
        input_xlsx=xlsx_path,
        output_xlsx=output_xlsx,
        image_paths=img_paths,
        plot_sheet=SHEET_PLOTS,
        start_cell="A1",
        row_step=44
    )

    # summary
    summary_to_write = summary_df.drop(columns=["|r|"], errors="ignore")
    write_df_to_sheet(output_xlsx, summary_to_write, SHEET_SUMMARY)
    save_table_csv(summary_to_write, dirs["tables"] / "summary.csv")

    # group stats
    if group_rows:
        gs = pd.concat(group_rows, axis=0, ignore_index=True)
        write_df_to_sheet(output_xlsx, gs, SHEET_GROUP_STATS)
        save_table_csv(gs, dirs["tables"] / "group_stats.csv")
    else:
        empty_gs = pd.DataFrame([{
            "note": "有効な種別列がない、または群別統計を作成できるデータがありません。"
        }])
        write_df_to_sheet(output_xlsx, empty_gs, SHEET_GROUP_STATS)
        save_table_csv(empty_gs, dirs["tables"] / "group_stats.csv")

    # outliers
    if outlier_tables:
        odf = pd.concat(outlier_tables, axis=0, ignore_index=True)

        cols_first = ["regression", "regression_label", "X", "Y", id_col]

        if group_col:
            cols_first.append(group_col)

        cols_first += [
            "slope",
            "intercept",
            "r",
            "diff_y_minus_x",
            "abs_diff_yx",
            "mean_xy",
            "rel_diff_yx_pct",
            "bias_pct_vs_x",
            "ratio_y_over_x",
            "pred_y",
            "residual",
            "abs_residual",
            "z_MAD",
            "abs_z_MAD",
            "outlier_level",
            "outlier_basis",
            "outlier_note"
        ]

        cols_first = [c for c in cols_first if c in odf.columns]
        rest_cols = [c for c in odf.columns if c not in cols_first]

        odf = odf[cols_first + rest_cols]

        write_df_to_sheet(output_xlsx, odf, SHEET_OUTLIERS)
        save_table_csv(odf, dirs["tables"] / "outliers.csv")
    else:
        empty_out = pd.DataFrame([{
            "note": "指定した zMAD 閾値以上の乖離候補はありませんでした。"
        }])
        write_df_to_sheet(output_xlsx, empty_out, SHEET_OUTLIERS)
        save_table_csv(empty_out, dirs["tables"] / "outliers.csv")

    # sample metrics
    if sample_metric_tables:
        smdf = pd.concat(sample_metric_tables, axis=0, ignore_index=True)

        cols_first = ["regression", "regression_label", "X", "Y", id_col]

        if group_col:
            cols_first.append(group_col)

        metric_cols = [
            "slope",
            "intercept",
            "r",
            "slope_95CI_low",
            "slope_95CI_high",
            "intercept_95CI_low",
            "intercept_95CI_high",
            "diff_y_minus_x",
            "abs_diff_yx",
            "mean_xy",
            "rel_diff_yx_pct",
            "bias_pct_vs_x",
            "ratio_y_over_x",
            "pred_y",
            "residual",
            "abs_residual",
            "z_MAD",
            "abs_z_MAD",
            "outlier_level",
            "outlier_z_thresh",
            "outlier_basis",
            "outlier_note",
            "outlier_excluded_from_fit",
            "fit_data_note",
            "range_filter_used",
            "range_min",
            "range_max",
            "range_condition"
        ]

        cols_first += metric_cols
        cols_first = [c for c in cols_first if c in smdf.columns]

        rest_cols = [c for c in smdf.columns if c not in cols_first]

        smdf = smdf[cols_first + rest_cols]

        write_df_to_sheet(output_xlsx, smdf, SHEET_SAMPLE_METRICS)
        save_table_csv(smdf, dirs["tables"] / "sample_metrics.csv")
    else:
        empty_smdf = pd.DataFrame([{
            "note": "sample_metricsを作成できるデータがありません。"
        }])
        save_table_csv(empty_smdf, dirs["tables"] / "sample_metrics.csv")

    # metric definitions
    metric_def_df = make_metric_definitions_df()
    write_df_to_sheet(output_xlsx, metric_def_df, SHEET_METRIC_DEFINITIONS)
    save_table_csv(metric_def_df, dirs["tables"] / "metric_definitions.csv")

    # run log
    log_text = f"""実行ログ

入力ファイル:
{xlsx_path}

入力シート:
{sheet_dd.value}

出力フォルダ:
{dirs["root"]}

出力Excel:
{output_xlsx}

モード:
{mode_dd.value}

ペア数:
{len(pairs)}

回帰法:
{"全回帰法" if all_reg_ck.value else REGRESSION_LABELS.get(reg_dd.value, reg_dd.value)}

Deming lambda:
{lam}

ベスト判定:
{best_label}

ベスト:
{best["regression_label"]} / {best["Y"]} vs {best["X"]}

ベスト n:
{int(best["n"])}

ベスト r:
{best["r"]}

ベスト slope:
{best["slope"]}

ベスト intercept:
{best["intercept"]}

対象範囲フィルタ:
{bool(use_range_ck.value)}

対象範囲:
{lo_range} ～ {hi_range}

範囲条件:
{range_mode_dd.value if use_range_ck.value else ""}

乖離zMAD閾値:
{z_thresh.value}

乖離IDラベル表示:
{bool(show_outlier_label_ck.value)}

乖離IDラベル数:
{label_top.value}

95%CI表示:
{bool(show_ci_ck.value)}
※95%CI表示はグラフ内テキスト表示のON/OFFです。CI線・CI帯は描画していません。

通常点サイズ:
{normal_point_size.value}

乖離○サイズ:
{outlier_point_size.value}

乖離○線幅:
{outlier_line_width.value}

図サイズ:
{fig_width_slider.value} x {fig_height_slider.value}

DPI:
{plot_dpi_slider.value}

Python表示枚数:
{shown}

備考:
相関係数・回帰係数は、乖離候補を除外せず、欠損除外・対象範囲フィルタ後の全データで算出しています。
"""

    with open(dirs["logs"] / "run_log.txt", "w", encoding="utf-8") as f:
        f.write(log_text)

    with status:
        print("\n完了！")
        print("出力Excel:", output_xlsx)
        print("画像フォルダ:", dirs["plots"])
        print("表フォルダ:", dirs["tables"])
        print("ログフォルダ:", dirs["logs"])
        print(
            f"シート: {SHEET_PLOTS}, {SHEET_SUMMARY}, {SHEET_GROUP_STATS}, "
            f"{SHEET_OUTLIERS}, {SHEET_SAMPLE_METRICS}, {SHEET_METRIC_DEFINITIONS}"
        )
        print("※Excelのシート構成・主要内容は従来どおりです。")
        print("※tables/logsは補助出力です。Excel内の解析結果を壊しません。")
        print("※乖離候補は自動除外していません。")
        print("※r/slope/interceptは、欠損除外・対象範囲フィルタ後の全データで算出しています。")


run_btn.on_click(run_all)

In [ ]:
import io
import re
import itertools
import math
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm

import ipywidgets as widgets
from IPython.display import display, clear_output

from openpyxl import load_workbook
from openpyxl.drawing.image import Image as XLImage


# ============================================================
# 0) 固定設定
# ============================================================
OUTPUT_ROOT = Path.cwd() / "data" / "export"

SHEET_PLOTS = "plots"
SHEET_SUMMARY = "summary"
SHEET_GROUP_STATS = "group_stats"
SHEET_OUTLIERS = "outliers"
SHEET_SAMPLE_METRICS = "sample_metrics"
SHEET_METRIC_DEFINITIONS = "metric_definitions"

OUT_SUFFIX = "_with_QCplots"

REGRESSION_METHODS_ALL = ["OLS", "Deming", "TheilSen", "PassingBablok"]

REGRESSION_LABELS = {
    "OLS": "OLS（最小二乗）",
    "Deming": "Deming（両軸誤差）",
    "TheilSen": "Theil-Sen（ロバスト）",
    "PassingBablok": "Passing-Bablok（ノンパラメトリック）"
}


# ============================================================
# 1) 日本語フォント
# ============================================================
def setup_japanese_font():
    candidates = [
        "IPAexGothic", "IPAPGothic",
        "Noto Sans CJK JP", "Noto Sans JP",
        "Yu Gothic", "YuGothic",
        "Meiryo", "MS Gothic",
        "Hiragino Sans", "TakaoGothic"
    ]

    available = {f.name for f in fm.fontManager.ttflist}
    chosen = None

    for c in candidates:
        if c in available:
            chosen = c
            break

    if chosen is None:
        for name in sorted(available):
            if any(k in name for k in ["IPA", "Noto", "Gothic", "Meiryo", "Hiragino", "ゴシック", "明朝"]):
                chosen = name
                break

    if chosen:
        plt.rcParams["font.family"] = chosen

    plt.rcParams["axes.unicode_minus"] = False


setup_japanese_font()


# ============================================================
# 2) 列推定
# ============================================================
ID_COL_CANDIDATES = ["検体ID", "SampleID", "Sample ID", "ID"]
GROUP_COL_CANDIDATES = ["種別", "Group", "Type", "分類"]
VALUE_PREFIXES = ("処方",)


def pick_col(df, candidates, default=None):
    for c in candidates:
        if c in df.columns:
            return c
    return default


def normalize_group_col(df, group_col):
    """
    種別列が実質的に使える場合だけ group_col を返す。
    使えない場合は None。
    """
    if group_col is None:
        return None

    if group_col not in df.columns:
        return None

    s = df[group_col]

    if s.notna().sum() == 0:
        return None

    nunique = s.nunique(dropna=True)

    if nunique <= 1:
        return None

    # 数値列でユニーク数が多い場合は分類列ではなく測定値列の可能性が高い
    if pd.api.types.is_numeric_dtype(s):
        n = s.notna().sum()
        if nunique > min(20, max(5, n // 3)):
            return None

    return group_col


def detect_value_cols(df, id_col, group_col, prefixes=("処方",)):
    """
    比較対象列を推定。
    まず「処方」で始まる列を優先し、なければ数値列からID列・種別列を除外。
    """
    cols = [c for c in df.columns if any(str(c).startswith(p) for p in prefixes)]

    if cols:
        return cols

    numeric_cols = df.select_dtypes(include="number").columns.tolist()

    exclude_cols = [id_col]
    if group_col:
        exclude_cols.append(group_col)

    return [c for c in numeric_cols if c not in exclude_cols]


def safe_name(s):
    s = str(s)
    return re.sub(r'[\\/:*?"<>|]', "_", s)


# ============================================================
# 3) FileUpload対応
# ============================================================
def _bytes_from_uploaded_content(content):
    if content is None:
        return None

    if isinstance(content, (bytes, bytearray)):
        return bytes(content)

    if hasattr(content, "tobytes"):
        return content.tobytes()

    return bytes(content)


def normalize_uploaded_item(uploader_value):
    v = uploader_value

    if not v:
        return None

    if isinstance(v, (list, tuple)):
        return v[0]

    if isinstance(v, dict):
        return list(v.values())[0]

    raise TypeError(f"FileUpload.value の型が想定外: {type(v)}")


def save_uploaded_xlsx(uploader_value, save_as="uploaded_input.xlsx"):
    item = normalize_uploaded_item(uploader_value)

    if item is None:
        return None

    if isinstance(item, dict):
        content = item.get("content", None)
    else:
        try:
            content = item["content"]
        except Exception:
            content = getattr(item, "content", None)

    b = _bytes_from_uploaded_content(content)

    if b is None:
        raise KeyError("アップロードから content を取得できませんでした。")

    path = Path(save_as)

    with open(path, "wb") as f:
        f.write(b)

    return path


# ============================================================
# 4) 出力フォルダ
# ============================================================
def make_output_dirs(output_root=OUTPUT_ROOT):
    root = Path(output_root)
    excel_dir = root / "excel"
    plot_dir = root / "plots_png"
    table_dir = root / "tables"
    log_dir = root / "logs"

    for d in [root, excel_dir, plot_dir, table_dir, log_dir]:
        d.mkdir(parents=True, exist_ok=True)

    return {
        "root": root,
        "excel": excel_dir,
        "plots": plot_dir,
        "tables": table_dir,
        "logs": log_dir
    }


# ============================================================
# 5) 対象値範囲フィルタ
# ============================================================
def apply_value_range_filter(
    df,
    xcol,
    ycol,
    use_range=False,
    lo=None,
    hi=None,
    mode="both"
):
    """
    X/Yの値で解析対象を絞る。

    mode:
      both   : XとYの両方が範囲内
      either : XまたはYが範囲内
      x_only : Xのみ範囲内
      y_only : Yのみ範囲内
    """
    if not use_range:
        return df.copy()

    if lo is None or hi is None:
        return df.copy()

    if lo > hi:
        lo, hi = hi, lo

    x = pd.to_numeric(df[xcol], errors="coerce")
    y = pd.to_numeric(df[ycol], errors="coerce")

    mx = pd.Series(True, index=df.index)
    my = pd.Series(True, index=df.index)

    mx &= x >= lo
    mx &= x <= hi
    my &= y >= lo
    my &= y <= hi

    if mode == "both":
        m = mx & my
    elif mode == "either":
        m = mx | my
    elif mode == "x_only":
        m = mx
    elif mode == "y_only":
        m = my
    else:
        m = mx & my

    return df.loc[m].copy()


# ============================================================
# 6) 統計・回帰
# ============================================================
def pearson_r(x, y):
    x = np.asarray(x, float)
    y = np.asarray(y, float)

    m = np.isfinite(x) & np.isfinite(y)
    x, y = x[m], y[m]

    if len(x) < 2:
        return np.nan

    return float(np.corrcoef(x, y)[0, 1])


def regression_fit(
    x,
    y,
    method="OLS",
    deming_lambda=1.0,
    theilsen_max_pairs=40000,
    seed=0
):
    x = np.asarray(x, float)
    y = np.asarray(y, float)

    m = np.isfinite(x) & np.isfinite(y)
    x, y = x[m], y[m]
    n = len(x)

    if n < 2:
        return np.nan, np.nan

    if method == "OLS":
        a, b = np.polyfit(x, y, 1)
        return float(a), float(b)

    if method == "Deming":
        lx = float(deming_lambda) if deming_lambda else 1.0

        xbar = x.mean()
        ybar = y.mean()

        sxx = np.mean((x - xbar) ** 2)
        syy = np.mean((y - ybar) ** 2)
        sxy = np.mean((x - xbar) * (y - ybar))

        if sxy == 0:
            a = 0.0
        else:
            a = (
                syy - lx * sxx
                + math.sqrt((syy - lx * sxx) ** 2 + 4 * lx * sxy * sxy)
            ) / (2 * sxy)

        b = ybar - a * xbar

        return float(a), float(b)

    if method == "TheilSen":
        rng = np.random.default_rng(seed)
        idx = np.arange(n)
        total_pairs = n * (n - 1) // 2
        slopes = []

        if total_pairs <= theilsen_max_pairs:
            for i in range(n - 1):
                dx = x[i + 1:] - x[i]
                dy = y[i + 1:] - y[i]

                mask = dx != 0

                slopes.extend((dy[mask] / dx[mask]).tolist())
        else:
            for _ in range(theilsen_max_pairs):
                i, j = rng.choice(idx, 2, replace=False)
                dx = x[j] - x[i]

                if dx == 0:
                    continue

                slopes.append((y[j] - y[i]) / dx)

        a = float(np.median(slopes)) if slopes else 0.0
        b = float(np.median(y - a * x))

        return a, b

    a, b = np.polyfit(x, y, 1)
    return float(a), float(b)


def passing_bablok_fit_with_ci(
    x,
    y,
    alpha=0.05,
    max_pairs=200000,
    seed=0
):
    """
    Passing-Bablok regression.
    CIは順位ベース近似。
    """
    x = np.asarray(x, float)
    y = np.asarray(y, float)

    m = np.isfinite(x) & np.isfinite(y)
    x, y = x[m], y[m]
    n = len(x)

    empty_info = {
        "slope_ci_low": np.nan,
        "slope_ci_high": np.nan,
        "intercept_ci_low": np.nan,
        "intercept_ci_high": np.nan,
        "pb_n_slopes": 0,
        "pb_ci_method": ""
    }

    if n < 3:
        empty_info["pb_ci_method"] = "insufficient n"
        return np.nan, np.nan, empty_info

    rng = np.random.default_rng(seed)
    total_pairs = n * (n - 1) // 2
    slopes = []

    if total_pairs <= max_pairs:
        for i in range(n - 1):
            dx = x[i + 1:] - x[i]
            dy = y[i + 1:] - y[i]

            mask = dx != 0
            s = dy[mask] / dx[mask]
            s = s[np.isfinite(s)]
            s = s[s != -1]

            slopes.extend(s.tolist())

        sampled = False
    else:
        idx = np.arange(n)

        for _ in range(max_pairs):
            i, j = rng.choice(idx, 2, replace=False)
            dx = x[j] - x[i]

            if dx == 0:
                continue

            s = (y[j] - y[i]) / dx

            if np.isfinite(s) and s != -1:
                slopes.append(float(s))

        sampled = True

    if len(slopes) == 0:
        empty_info["pb_ci_method"] = "no valid slopes"
        return np.nan, np.nan, empty_info

    slopes = np.sort(np.asarray(slopes, dtype=float))
    N = len(slopes)
    K = int(np.sum(slopes < -1))

    if N % 2 == 1:
        idx_med = (N - 1) // 2 + K
        idx_med = max(0, min(idx_med, N - 1))
        slope = float(slopes[idx_med])
    else:
        idx1 = N // 2 - 1 + K
        idx2 = N // 2 + K

        idx1 = max(0, min(idx1, N - 1))
        idx2 = max(0, min(idx2, N - 1))

        slope = float((slopes[idx1] + slopes[idx2]) / 2)

    intercept = float(np.median(y - slope * x))

    z = 1.959963984540054
    C = z * math.sqrt(n * (n - 1) * (2 * n + 5) / 18)

    m1 = int(math.floor((N - C) / 2 + K))
    m2 = int(math.ceil((N + C) / 2 + K))

    m1 = max(0, min(m1, N - 1))
    m2 = max(0, min(m2, N - 1))

    slope_low = float(slopes[min(m1, m2)])
    slope_high = float(slopes[max(m1, m2)])

    int_candidates = [
        np.median(y - slope_low * x),
        np.median(y - slope_high * x)
    ]

    intercept_low = float(np.nanmin(int_candidates))
    intercept_high = float(np.nanmax(int_candidates))

    info = {
        "slope_ci_low": slope_low,
        "slope_ci_high": slope_high,
        "intercept_ci_low": intercept_low,
        "intercept_ci_high": intercept_high,
        "pb_n_slopes": int(N),
        "pb_ci_method": "approx_rank_based_sampled" if sampled else "approx_rank_based_all_pairs"
    }

    return slope, intercept, info


def regression_fit_info(
    x,
    y,
    method="OLS",
    deming_lambda=1.0,
    theilsen_max_pairs=40000,
    seed=0
):
    info = {
        "slope_ci_low": np.nan,
        "slope_ci_high": np.nan,
        "intercept_ci_low": np.nan,
        "intercept_ci_high": np.nan,
        "pb_n_slopes": np.nan,
        "pb_ci_method": ""
    }

    if method == "PassingBablok":
        a, b, pb_info = passing_bablok_fit_with_ci(
            x,
            y,
            alpha=0.05,
            max_pairs=200000,
            seed=seed
        )

        info.update(pb_info)

        return a, b, info

    a, b = regression_fit(
        x,
        y,
        method=method,
        deming_lambda=deming_lambda,
        theilsen_max_pairs=theilsen_max_pairs,
        seed=seed
    )

    return a, b, info


# ============================================================
# 7) 乖離候補・検体別指標
# ============================================================
def mad(arr):
    arr = np.asarray(arr, float)
    med = np.nanmedian(arr)

    return np.nanmedian(np.abs(arr - med))


def compute_residuals(x, y, a, b):
    x = np.asarray(x, float)
    y = np.asarray(y, float)

    return y - (a * x + b)


def classify_outlier_level(abs_z):
    if not np.isfinite(abs_z):
        return "not_evaluable"

    if abs_z >= 5.0:
        return "strong_candidate"

    if abs_z >= 3.5:
        return "candidate"

    if abs_z >= 2.5:
        return "mild_candidate"

    return "none"


def compute_pair_sample_metrics(df, id_col, group_col, xcol, ycol, a, b):
    has_group = group_col and group_col in df.columns

    cols = [id_col, xcol, ycol] + ([group_col] if has_group else [])

    sub = df[cols].dropna(subset=[xcol, ycol]).copy()

    if sub.empty:
        return sub

    x = pd.to_numeric(sub[xcol], errors="coerce").astype(float)
    y = pd.to_numeric(sub[ycol], errors="coerce").astype(float)

    sub["X"] = xcol
    sub["Y"] = ycol

    sub["diff_y_minus_x"] = y - x
    sub["abs_diff_yx"] = np.abs(sub["diff_y_minus_x"])
    sub["mean_xy"] = (x + y) / 2.0

    sub["rel_diff_yx_pct"] = np.where(
        sub["mean_xy"] != 0,
        sub["abs_diff_yx"] / np.abs(sub["mean_xy"]) * 100,
        np.nan
    )

    sub["bias_pct_vs_x"] = np.where(
        x != 0,
        (y - x) / np.abs(x) * 100,
        np.nan
    )

    sub["ratio_y_over_x"] = np.where(
        x != 0,
        y / x,
        np.nan
    )

    sub["pred_y"] = a * x + b
    sub["residual"] = y - sub["pred_y"]
    sub["abs_residual"] = np.abs(sub["residual"])

    resid = sub["residual"].to_numpy(dtype=float)
    s = mad(resid)
    scale = 1.4826 * s if (np.isfinite(s) and s > 0) else np.nan

    if np.isfinite(scale):
        sub["z_MAD"] = sub["residual"] / scale
    else:
        sub["z_MAD"] = np.nan

    sub["abs_z_MAD"] = np.abs(sub["z_MAD"])
    sub["outlier_level"] = sub["abs_z_MAD"].apply(classify_outlier_level)

    sub["outlier_basis"] = "回帰残差 y-(slope*x+intercept) をMADで標準化"
    sub["outlier_note"] = "乖離候補であり、自動除外ではない"

    return sub


def pick_outliers_table(
    df,
    id_col,
    group_col,
    xcol,
    ycol,
    a,
    b,
    z_thresh=3.5,
    top_n=200
):
    sub = compute_pair_sample_metrics(
        df=df,
        id_col=id_col,
        group_col=group_col,
        xcol=xcol,
        ycol=ycol,
        a=a,
        b=b
    )

    if sub.empty or "abs_z_MAD" not in sub.columns:
        return sub, sub

    flagged = sub[
        np.isfinite(sub["abs_z_MAD"])
        & (sub["abs_z_MAD"] >= z_thresh)
    ].copy()

    flagged = flagged.sort_values("abs_z_MAD", ascending=False).head(top_n)

    return sub, flagged


# ============================================================
# 8) プロット作成
# ============================================================
def plot_suite(
    df,
    id_col,
    group_col,
    xcol,
    ycol,
    method,
    lam,
    a,
    b,
    r,
    fit_info=None,
    z_thresh=3.5,
    outlier_label_top=8,
    add_diag=True,
    show_fit=True,
    show_stats=True,
    show_equation=True,
    show_ci=True,
    show_ba_text=True,
    show_outlier_text=True,
    normal_s=16,
    outlier_s=28,
    outlier_lw=1.2,
    fig_width=16,
    fig_height=10,
    dpi=150
):
    if fit_info is None:
        fit_info = {}

    has_group = group_col and group_col in df.columns

    base_cols = [id_col, xcol, ycol] + ([group_col] if has_group else [])

    sub = df[base_cols].dropna(subset=[xcol, ycol]).copy()

    if sub.empty:
        return None, None, None, None, None

    all_rows, flagged = pick_outliers_table(
        df=df,
        id_col=id_col,
        group_col=group_col,
        xcol=xcol,
        ycol=ycol,
        a=a,
        b=b,
        z_thresh=z_thresh,
        top_n=200
    )

    x = pd.to_numeric(sub[xcol], errors="coerce").astype(float).to_numpy()
    y = pd.to_numeric(sub[ycol], errors="coerce").astype(float).to_numpy()

    resid = compute_residuals(x, y, a, b)
    mean_xy = (x + y) / 2.0
    diff_yx = y - x

    bias = float(np.nanmean(diff_yx))
    sd = float(np.nanstd(diff_yx, ddof=1)) if len(diff_yx) > 1 else np.nan

    loa_hi = bias + 1.96 * sd if np.isfinite(sd) else np.nan
    loa_lo = bias - 1.96 * sd if np.isfinite(sd) else np.nan

    flagged_ids = set(flagged[id_col].tolist()) if flagged is not None and not flagged.empty else set()
    is_flagged = sub[id_col].isin(flagged_ids).to_numpy()

    if has_group:
        groups = sorted(sub[group_col].dropna().unique().tolist(), key=lambda t: str(t))
        cmap = plt.get_cmap("tab10")
        color_map = {g: cmap(i % 10) for i, g in enumerate(groups)}
        colors = sub[group_col].map(lambda g: color_map.get(g, "gray")).to_numpy()
    else:
        groups = []
        color_map = {}
        colors = np.array(["C0"] * len(sub))

    fig = plt.figure(figsize=(fig_width, fig_height), dpi=dpi)
    gs = fig.add_gridspec(2, 2)

    # ---------------- Scatter ----------------
    ax1 = fig.add_subplot(gs[0, 0])

    ax1.scatter(
        x[~is_flagged],
        y[~is_flagged],
        s=normal_s,
        alpha=0.75,
        c=colors[~is_flagged]
    )

    if np.any(is_flagged):
        ax1.scatter(
            x[is_flagged],
            y[is_flagged],
            s=outlier_s,
            alpha=0.95,
            facecolors="none",
            edgecolors="red",
            linewidths=outlier_lw
        )

    ax1.set_title(f"散布図：{ycol} vs {xcol} / {REGRESSION_LABELS.get(method, method)}")
    ax1.set_xlabel(xcol)
    ax1.set_ylabel(ycol)
    ax1.grid(True, alpha=0.25)

    lo_xy = float(np.nanmin([np.nanmin(x), np.nanmin(y)]))
    hi_xy = float(np.nanmax([np.nanmax(x), np.nanmax(y)]))

    if add_diag:
        ax1.plot(
            [lo_xy, hi_xy],
            [lo_xy, hi_xy],
            "--",
            lw=1,
            alpha=0.6,
            label="y=x"
        )

    if show_fit and np.isfinite(a) and np.isfinite(b):
        xx = np.array([lo_xy, hi_xy])
        ax1.plot(
            xx,
            a * xx + b,
            lw=1.8,
            alpha=0.85,
            label="回帰直線"
        )

    if show_stats:
        lines = []
        lines.append(f"n={len(x)}")
        lines.append(f"Pearson r={r:.4f}")
        lines.append(
            REGRESSION_LABELS.get(method, method)
            + (f" (λ={lam:g})" if method == "Deming" else "")
        )

        if show_equation:
            lines.append(f"y={a:.4f}x+{b:.4f}")

        if show_ci:
            sl = fit_info.get("slope_ci_low", np.nan)
            sh = fit_info.get("slope_ci_high", np.nan)
            il = fit_info.get("intercept_ci_low", np.nan)
            ih = fit_info.get("intercept_ci_high", np.nan)

            if np.isfinite(sl) and np.isfinite(sh):
                lines.append(f"slope 95%CI [{sl:.4f}, {sh:.4f}]")

            if np.isfinite(il) and np.isfinite(ih):
                lines.append(f"intercept 95%CI [{il:.4f}, {ih:.4f}]")

        if show_ba_text:
            lines.append(f"BA bias={bias:.4g}")

            if np.isfinite(loa_lo) and np.isfinite(loa_hi):
                lines.append(f"LoA [{loa_lo:.4g}, {loa_hi:.4g}]")

        if show_outlier_text:
            n_flag = int(len(flagged)) if flagged is not None else 0
            lines.append(f"乖離候補: |z_MAD|≥{z_thresh}")
            lines.append(f"乖離候補数={n_flag}")

        ax1.text(
            0.03,
            0.97,
            "\n".join(lines),
            transform=ax1.transAxes,
            va="top",
            fontsize=9,
            bbox=dict(boxstyle="round", facecolor="white", alpha=0.75)
        )

    # ---------------- 乖離IDラベル ----------------
    # 修正済み：
    # 以前の row(row[xcol], row[ycol]) は pandas.Series を関数として呼んでしまうためエラー。
    # flagged には xcol/ycol の値が含まれているので、ここでは直接 row[xcol], row[ycol] を使う。
    if (
        np.any(is_flagged)
        and outlier_label_top > 0
        and flagged is not None
        and not flagged.empty
    ):
        top_flagged = flagged.sort_values("abs_z_MAD", ascending=False).head(outlier_label_top)

        for _, row in top_flagged.iterrows():
            try:
                xx0 = float(row[xcol])
                yy0 = float(row[ycol])
                sid = row[id_col]

                if np.isfinite(xx0) and np.isfinite(yy0):
                    ax1.text(
                        xx0,
                        yy0,
                        str(sid),
                        fontsize=8,
                        color="red"
                    )
            except Exception:
                continue

    if has_group:
        for g in groups[:10]:
            ax1.scatter([], [], c=[color_map[g]], label=f"{group_col}:{g}")

    ax1.legend(fontsize=8, loc="best")

    # ---------------- Bland-Altman ----------------
    ax2 = fig.add_subplot(gs[0, 1])

    ax2.scatter(
        mean_xy[~is_flagged],
        diff_yx[~is_flagged],
        s=normal_s,
        alpha=0.75,
        c=colors[~is_flagged]
    )

    if np.any(is_flagged):
        ax2.scatter(
            mean_xy[is_flagged],
            diff_yx[is_flagged],
            s=outlier_s,
            alpha=0.95,
            facecolors="none",
            edgecolors="red",
            linewidths=outlier_lw
        )

    ax2.axhline(
        bias,
        color="black",
        lw=1.5,
        label=f"平均差={bias:.3g}"
    )

    if np.isfinite(loa_hi) and np.isfinite(loa_lo):
        ax2.axhline(
            loa_hi,
            color="gray",
            lw=1.2,
            ls="--",
            label=f"+1.96SD={loa_hi:.3g}"
        )

        ax2.axhline(
            loa_lo,
            color="gray",
            lw=1.2,
            ls="--",
            label=f"-1.96SD={loa_lo:.3g}"
        )

    ax2.set_title("Bland–Altman：差(Y-X) vs 平均((X+Y)/2)")
    ax2.set_xlabel("平均 (X+Y)/2")
    ax2.set_ylabel("差 (Y-X)")
    ax2.grid(True, alpha=0.25)
    ax2.legend(fontsize=8, loc="best")

    # ---------------- Residuals ----------------
    ax3 = fig.add_subplot(gs[1, 0])

    ax3.scatter(
        x[~is_flagged],
        resid[~is_flagged],
        s=normal_s,
        alpha=0.75,
        c=colors[~is_flagged]
    )

    if np.any(is_flagged):
        ax3.scatter(
            x[is_flagged],
            resid[is_flagged],
            s=outlier_s,
            alpha=0.95,
            facecolors="none",
            edgecolors="red",
            linewidths=outlier_lw
        )

    ax3.axhline(0, color="black", lw=1.2)
    ax3.set_title("残差プロット：残差(Y-(aX+b)) vs X")
    ax3.set_xlabel(xcol)
    ax3.set_ylabel("残差")
    ax3.grid(True, alpha=0.25)

    # ---------------- Text panel ----------------
    ax4 = fig.add_subplot(gs[1, 1])
    ax4.axis("off")

    n_flagged = int(len(flagged)) if flagged is not None else 0

    txt = (
        f"【統計まとめ】\n"
        f"X={xcol}\n"
        f"Y={ycol}\n"
        f"n={len(x)}\n"
        f"Pearson r={r:.6f}\n"
        f"回帰: {REGRESSION_LABELS.get(method, method)}"
        + (f" (λ={lam:g})" if method == "Deming" else "")
        + "\n"
        f"slope={a:.6f}\n"
        f"intercept={b:.6f}\n"
    )

    sl = fit_info.get("slope_ci_low", np.nan)
    sh = fit_info.get("slope_ci_high", np.nan)
    il = fit_info.get("intercept_ci_low", np.nan)
    ih = fit_info.get("intercept_ci_high", np.nan)

    if np.isfinite(sl) and np.isfinite(sh):
        txt += f"slope 95%CI=[{sl:.6g}, {sh:.6g}]\n"

    if np.isfinite(il) and np.isfinite(ih):
        txt += f"intercept 95%CI=[{il:.6g}, {ih:.6g}]\n"

    txt += (
        f"\n【Bland–Altman】\n"
        f"bias(Y-X)={bias:.6g}\n"
    )

    if np.isfinite(sd):
        txt += f"SD={sd:.6g}\nLoA=[{loa_lo:.6g}, {loa_hi:.6g}]\n"

    txt += (
        f"\n【乖離候補】\n"
        f"|z_MAD|≥{z_thresh}\n"
        f"乖離候補数={n_flagged}\n"
        f"※乖離候補は自動除外していません\n"
        f"※r/slope/interceptは解析対象全データで算出\n"
    )

    ax4.text(
        0.02,
        0.98,
        txt,
        va="top",
        fontsize=10
    )

    fig.tight_layout()

    return fig, sub, flagged, bias, (loa_lo, loa_hi)


# ============================================================
# 9) Excel出力
# ============================================================
def _excel_safe_value(v):
    if isinstance(v, (np.integer,)):
        return int(v)

    if isinstance(v, (np.floating,)):
        if np.isnan(v) or np.isinf(v):
            return None
        return float(v)

    if isinstance(v, float):
        if math.isnan(v) or math.isinf(v):
            return None
        return v

    if pd.isna(v):
        return None

    return v


def clean_df_for_excel(df):
    if df is None:
        return pd.DataFrame()

    out = df.copy()
    out = out.replace([np.inf, -np.inf], np.nan)
    out = out.where(pd.notna(out), None)

    return out


def insert_images_into_excel(
    input_xlsx,
    output_xlsx,
    image_paths,
    plot_sheet=SHEET_PLOTS,
    start_cell="A1",
    row_step=44,
    max_width_px=1100
):
    wb = load_workbook(input_xlsx)

    if plot_sheet in wb.sheetnames:
        wb.remove(wb[plot_sheet])

    ws = wb.create_sheet(plot_sheet)

    m = re.match(r"([A-Z]+)(\d+)", start_cell)

    if not m:
        start_cell = "A1"
        m = re.match(r"([A-Z]+)(\d+)", start_cell)

    col_letters = m.group(1)
    row = int(m.group(2))

    for p in image_paths:
        img = XLImage(str(p))

        if img.width and img.width > max_width_px:
            scale = max_width_px / img.width
            img.width = int(img.width * scale)
            img.height = int(img.height * scale)

        ws.add_image(img, f"{col_letters}{row}")
        row += row_step

    wb.save(output_xlsx)


def write_df_to_sheet(xlsx_path, df, sheet_name, index=False):
    df = clean_df_for_excel(df)

    wb = load_workbook(xlsx_path)

    if sheet_name in wb.sheetnames:
        wb.remove(wb[sheet_name])

    ws = wb.create_sheet(sheet_name)

    df_to_write = df.reset_index() if index else df

    ws.append(list(df_to_write.columns))

    for row in df_to_write.itertuples(index=False):
        ws.append([_excel_safe_value(v) for v in row])

    wb.save(xlsx_path)


# ============================================================
# 10) 種別ごとの統計
# ============================================================
def groupwise_stats(df, id_col, group_col, xcol, ycol, method, lam):
    if not (group_col and group_col in df.columns):
        return pd.DataFrame(columns=[
            "X",
            "Y",
            "group",
            "n",
            "r",
            "slope",
            "intercept",
            "regression",
            "lambda"
        ])

    rows = []

    sub = df[[group_col, xcol, ycol]].dropna(subset=[xcol, ycol]).copy()

    for g, gdf in sub.groupby(group_col, dropna=False):
        x = pd.to_numeric(gdf[xcol], errors="coerce").astype(float).to_numpy()
        y = pd.to_numeric(gdf[ycol], errors="coerce").astype(float).to_numpy()

        if len(x) < 2:
            continue

        r = pearson_r(x, y)
        a, b, fit_info = regression_fit_info(
            x,
            y,
            method=method,
            deming_lambda=lam
        )

        rows.append({
            "X": xcol,
            "Y": ycol,
            "group": str(g),
            "n": len(x),
            "r": r,
            "slope": a,
            "intercept": b,
            "regression": method,
            "regression_label": REGRESSION_LABELS.get(method, method),
            "lambda": lam if method == "Deming" else np.nan,
            "slope_95CI_low": fit_info.get("slope_ci_low", np.nan),
            "slope_95CI_high": fit_info.get("slope_ci_high", np.nan),
            "intercept_95CI_low": fit_info.get("intercept_ci_low", np.nan),
            "intercept_95CI_high": fit_info.get("intercept_ci_high", np.nan),
            "outlier_excluded_from_fit": False,
            "fit_data_note": "群別統計も乖離候補を除外せず算出"
        })

    return pd.DataFrame(rows)


# ============================================================
# 11) 指標説明シート
# ============================================================
def make_metric_definitions_df():
    rows = [
        [
            "入力データ",
            "検体ID / SampleID / ID",
            "各検体を識別する列",
            "入力列",
            "図中IDラベル、outliers、sample_metricsで検体を追跡",
            "列名がない場合は先頭列をID列として扱います"
        ],
        [
            "入力データ",
            "種別 / Group / Type",
            "任意の分類列",
            "入力列",
            "存在し、分類列と判断できる場合のみ色分け・群別統計に使用",
            "なくても動きます"
        ],
        [
            "入力データ",
            "処方列",
            "比較対象となる数値列",
            "入力列",
            "列名が「処方」で始まる列を優先。なければ数値列を候補",
            "2列以上必要です"
        ],
        [
            "基本情報",
            "X",
            "比較ペアにおける基準側の列名",
            "列名",
            "例：X=処方1",
            ""
        ],
        [
            "基本情報",
            "Y",
            "比較ペアにおける比較側の列名",
            "列名",
            "例：Y=処方2",
            ""
        ],
        [
            "差分",
            "diff_y_minus_x",
            "YとXの単純差",
            "Y - X",
            "正ならYがXより高く、負なら低い",
            "Bland–Altmanの差と同じです"
        ],
        [
            "差分",
            "abs_diff_yx",
            "YとXの差の絶対値",
            "|Y - X|",
            "方向を問わず離れ具合を見る",
            "単位依存です"
        ],
        [
            "Bland–Altman",
            "mean_xy",
            "XとYの平均値",
            "(X + Y) / 2",
            "BAプロットの横軸",
            ""
        ],
        [
            "相対差",
            "rel_diff_yx_pct",
            "平均値基準の相対差",
            "|Y - X| / |(X+Y)/2| × 100",
            "値域差をならしたズレ",
            "平均が0に近いと不安定"
        ],
        [
            "相対差",
            "bias_pct_vs_x",
            "X基準の差の割合",
            "(Y - X) / |X| × 100",
            "Xに対してYが何%高い/低いか",
            "Xが0に近いと不安定"
        ],
        [
            "比率",
            "ratio_y_over_x",
            "Y/X比",
            "Y / X",
            "1に近いほど近い",
            "Xが0に近いと不安定"
        ],
        [
            "回帰",
            "pred_y",
            "回帰式から予測されるY",
            "slope × X + intercept",
            "全体傾向から期待されるY",
            ""
        ],
        [
            "回帰残差",
            "residual",
            "実測Yと予測Yの差",
            "Y - pred_y",
            "回帰直線からのズレ",
            "単純なY-Xではありません"
        ],
        [
            "回帰残差",
            "abs_residual",
            "残差の絶対値",
            "|Y - pred_y|",
            "方向を問わず回帰からの外れ具合",
            "単位依存です"
        ],
        [
            "乖離候補",
            "z_MAD",
            "残差をMADで標準化した指標",
            "residual / (1.4826 × MAD(residual))",
            "全体の残差ばらつきに対する外れ具合",
            "自動除外基準ではありません"
        ],
        [
            "乖離候補",
            "abs_z_MAD",
            "z_MADの絶対値",
            "|z_MAD|",
            "大きいほど乖離候補",
            "方向はz_MADまたはresidualで確認"
        ],
        [
            "乖離候補",
            "outlier_level",
            "確認優先度",
            "2.5以上=mild, 3.5以上=candidate, 5.0以上=strong",
            "乖離候補の段階表示",
            "除外判定ではありません"
        ],
        [
            "回帰統計",
            "r",
            "Pearson相関係数",
            "corr(X, Y)",
            "1に近いほど正の直線相関が強い",
            "一致性ではありません"
        ],
        [
            "回帰統計",
            "slope",
            "回帰直線の傾き",
            "回帰法に依存",
            "1に近いほど比例性が近い",
            "回帰法により変わります"
        ],
        [
            "回帰統計",
            "intercept",
            "回帰直線の切片",
            "回帰法に依存",
            "0に近いほど定数差が小さい",
            "測定範囲外の解釈注意"
        ],
        [
            "Bland–Altman",
            "BA_bias(y-x)",
            "Y-Xの平均差",
            "mean(Y - X)",
            "平均的な差",
            "外れ値の影響を受けます"
        ],
        [
            "Bland–Altman",
            "BA_LoA_low / BA_LoA_high",
            "95%一致限界",
            "bias ± 1.96×SD(Y-X)",
            "差が多く入る想定範囲",
            "差の分布に依存"
        ],
        [
            "仕様",
            "outlier_excluded_from_fit",
            "乖離候補を回帰計算から除外したか",
            "False=除外していない",
            "r/slope/interceptは解析対象全データで算出",
            "対象範囲フィルタで外れた値は除外"
        ]
    ]

    return pd.DataFrame(
        rows,
        columns=["分類", "指標名", "意味", "計算式", "解釈", "注意点"]
    )


# ============================================================
# 12) UI
# ============================================================
help_html = widgets.HTML("""
<div style="border:1px solid #ccc; padding:12px; border-radius:8px; background:#fafafa; line-height:1.6;">
<b>このツールの目的</b><br>
処方列どうしの相関・回帰・Bland–Altman・残差・乖離候補をまとめて確認し、Excelに出力します。<br>
乖離候補は自動除外ではなく、確認すべき候補として出力します。<br><br>

<b>必要な入力データ</b><br>
<ul>
<li><b>検体ID列</b>：推奨列名は「検体ID」「SampleID」「ID」です。ない場合は先頭列をIDとして扱います。</li>
<li><b>処方列</b>：比較したい数値列です。列名が「処方」で始まる列を優先して検出します。</li>
<li><b>種別列</b>：任意です。「種別」「Group」「Type」があれば群別色分け・群別統計に使います。なくても動きます。</li>
</ul>

<b>主な設定値</b><br>
<ul>
<li><b>全回帰法で出力</b>：ONにすると OLS / Deming / Theil-Sen / Passing-Bablok の全てで図・統計を出力します。</li>
<li><b>回帰法</b>：全回帰法OFFのときに使用する回帰法です。</li>
<li><b>ベスト</b>：初期設定は r最大（正相関優先）です。</li>
<li><b>対象範囲</b>：下限・上限に数値を入力します。通常はOFFで良いです。</li>
<li><b>乖離z(MAD)</b>：回帰残差をMADで標準化した値です。乖離候補の抽出に使います。</li>
<li><b>通常点サイズ / 乖離○サイズ</b>：プロットの見やすさ調整用です。</li>
</ul>

<b>重要</b><br>
相関係数・傾き・切片は、乖離候補を除外せずに算出します。<br>
除外するかどうかは、測定情報・入力情報・検体情報を確認してから判断してください。
</div>
""")

uploader = widgets.FileUpload(
    accept=".xlsx",
    multiple=False,
    description="Excelをアップロード"
)

sheet_dd = widgets.Dropdown(
    options=["(未選択)"],
    value="(未選択)",
    description="Sheet"
)

mode_dd = widgets.Dropdown(
    options=[
        ("全組合せ", "all"),
        ("隣同士のみ", "adjacent"),
        ("基準処方 vs その他", "baseline")
    ],
    value="baseline",
    description="モード"
)

baseline_sel = widgets.SelectMultiple(
    options=[],
    value=(),
    description="基準処方",
    rows=8
)

all_reg_ck = widgets.Checkbox(
    value=False,
    description="全回帰法で出力"
)

reg_dd = widgets.Dropdown(
    options=[
        ("OLS（最小二乗）", "OLS"),
        ("Deming（両軸誤差）", "Deming"),
        ("Theil-Sen（ロバスト）", "TheilSen"),
        ("Passing-Bablok（ノンパラメトリック）", "PassingBablok"),
    ],
    value="PassingBablok",
    description="回帰法"
)

deming_lambda = widgets.FloatText(
    value=1.0,
    description="λ(Deming)"
)

best_mode = widgets.ToggleButtons(
    options=[
        ("|r|最大", "abs"),
        ("r最大（正相関優先）", "pos")
    ],
    value="pos",
    description="ベスト"
)

z_thresh = widgets.FloatSlider(
    value=3.5,
    min=1.5,
    max=8.0,
    step=0.1,
    description="乖離z(MAD)"
)

label_top = widgets.IntSlider(
    value=8,
    min=0,
    max=30,
    step=1,
    description="乖離IDラベル数"
)

use_range_ck = widgets.Checkbox(
    value=False,
    description="対象範囲で絞る"
)

range_min_txt = widgets.FloatText(
    value=0.0,
    description="下限"
)

range_max_txt = widgets.FloatText(
    value=100.0,
    description="上限"
)

range_mode_dd = widgets.Dropdown(
    options=[
        ("XとYの両方が範囲内", "both"),
        ("XまたはYが範囲内", "either"),
        ("Xのみ範囲内", "x_only"),
        ("Yのみ範囲内", "y_only"),
    ],
    value="both",
    description="範囲条件"
)

add_diag_ck = widgets.Checkbox(
    value=True,
    description="y=x補助線"
)

show_fit_ck = widgets.Checkbox(
    value=True,
    description="回帰直線"
)

show_stats_ck = widgets.Checkbox(
    value=True,
    description="統計表示"
)

show_py_ck = widgets.Checkbox(
    value=True,
    description="Python上に表示"
)

max_show = widgets.IntSlider(
    value=6,
    min=0,
    max=30,
    step=1,
    description="表示上限"
)

show_equation_ck = widgets.Checkbox(
    value=True,
    description="回帰式を表示"
)

show_ci_ck = widgets.Checkbox(
    value=True,
    description="95%CIを表示"
)

show_ba_text_ck = widgets.Checkbox(
    value=True,
    description="BA統計を表示"
)

show_outlier_text_ck = widgets.Checkbox(
    value=True,
    description="乖離数を表示"
)

normal_point_size = widgets.IntSlider(
    value=16,
    min=4,
    max=80,
    step=2,
    description="通常点サイズ"
)

outlier_point_size = widgets.IntSlider(
    value=28,
    min=6,
    max=120,
    step=2,
    description="乖離○サイズ"
)

outlier_line_width = widgets.FloatSlider(
    value=1.2,
    min=0.5,
    max=4.0,
    step=0.1,
    description="乖離○線幅"
)

fig_width_slider = widgets.IntSlider(
    value=16,
    min=8,
    max=28,
    step=1,
    description="図横サイズ"
)

fig_height_slider = widgets.IntSlider(
    value=10,
    min=6,
    max=20,
    step=1,
    description="図縦サイズ"
)

plot_dpi_slider = widgets.IntSlider(
    value=150,
    min=80,
    max=300,
    step=10,
    description="DPI"
)

load_btn = widgets.Button(
    description="列を読み込み（基準候補更新）",
    button_style="info"
)

run_btn = widgets.Button(
    description="実行（解析→Excel出力）",
    button_style="success"
)

status = widgets.Output()
plots_out = widgets.Output()

display(widgets.VBox([
    widgets.HTML("<b>①アップロード → ②列読み込み → ③実行</b>"),
    help_html,
    uploader,
    sheet_dd,
    widgets.HBox([mode_dd, all_reg_ck, reg_dd, deming_lambda]),
    widgets.HBox([baseline_sel, widgets.VBox([best_mode, z_thresh, label_top])]),
    widgets.HBox([use_range_ck, range_min_txt, range_max_txt, range_mode_dd]),
    widgets.HBox([add_diag_ck, show_fit_ck, show_stats_ck, show_py_ck, max_show]),
    widgets.HBox([show_equation_ck, show_ci_ck, show_ba_text_ck, show_outlier_text_ck]),
    widgets.HBox([normal_point_size, outlier_point_size, outlier_line_width]),
    widgets.HBox([fig_width_slider, fig_height_slider, plot_dpi_slider]),
    widgets.HBox([load_btn, run_btn]),
    status,
    plots_out
]))

_state = {
    "xlsx_path": None,
    "df": None,
    "id_col": None,
    "group_col": None,
    "value_cols": None
}


# ============================================================
# 13) UI callbacks
# ============================================================
def refresh_sheet_list(xlsx_path):
    xl = pd.ExcelFile(xlsx_path, engine="openpyxl")
    sheets = xl.sheet_names

    if sheets:
        sheet_dd.options = sheets
        sheet_dd.value = sheets[0]
    else:
        sheet_dd.options = ["(未選択)"]
        sheet_dd.value = "(未選択)"


def set_default_range_from_data(df, value_cols):
    vals = []

    for c in value_cols:
        s = pd.to_numeric(df[c], errors="coerce")
        vals.append(s)

    if not vals:
        return

    allv = pd.concat(vals, axis=0).dropna()

    if allv.empty:
        return

    lo0 = float(np.nanmin(allv))
    hi0 = float(np.nanmax(allv))

    if np.isfinite(lo0):
        range_min_txt.value = lo0

    if np.isfinite(hi0):
        range_max_txt.value = hi0


def on_upload_change(change):
    with status:
        clear_output()

        try:
            xlsx_path = save_uploaded_xlsx(
                uploader.value,
                save_as="uploaded_input.xlsx"
            )
        except Exception as e:
            print("アップロード読み込みエラー:", repr(e))
            return

        if xlsx_path is None:
            print("Excelをアップロードしてください。")
            return

        _state["xlsx_path"] = xlsx_path

        print("アップロードOK:", xlsx_path)

        refresh_sheet_list(xlsx_path)

        print("Sheet:", sheet_dd.value)


uploader.observe(on_upload_change, names="value")


def load_columns(_=None):
    with status:
        clear_output()

        xlsx_path = _state.get("xlsx_path")

        if xlsx_path is None:
            print("先にExcelをアップロードしてください。")
            return

        if sheet_dd.value == "(未選択)":
            print("Sheetを選択してください。")
            return

        df = pd.read_excel(
            xlsx_path,
            sheet_name=sheet_dd.value,
            engine="openpyxl"
        )

        id_col = pick_col(
            df,
            ID_COL_CANDIDATES,
            default=df.columns[0]
        )

        group_col_raw = pick_col(
            df,
            GROUP_COL_CANDIDATES,
            default=None
        )

        group_col = normalize_group_col(df, group_col_raw)

        value_cols = detect_value_cols(
            df,
            id_col,
            group_col,
            prefixes=VALUE_PREFIXES
        )

        _state.update({
            "df": df,
            "id_col": id_col,
            "group_col": group_col,
            "value_cols": value_cols
        })

        baseline_sel.options = value_cols

        if value_cols:
            baseline_sel.value = (value_cols[0],)

        set_default_range_from_data(df, value_cols)

        print("列読み込み完了")
        print("  ID列   :", id_col)
        print("  種別列 :", group_col if group_col else "(なし)")
        print("  比較列 :", value_cols)
        print(f"  対象範囲初期値: {range_min_txt.value:.4g} ～ {range_max_txt.value:.4g}")

        if len(value_cols) < 2:
            print("注意：比較列が2列未満です。解析には2列以上必要です。")


load_btn.on_click(load_columns)


def run_all(_=None):
    with status:
        clear_output()

    with plots_out:
        clear_output()

    xlsx_path = _state.get("xlsx_path")

    if xlsx_path is None:
        with status:
            print("先にExcelをアップロードしてください。")
        return

    if _state.get("df") is None:
        load_columns()

    df = _state.get("df")
    id_col = _state.get("id_col")
    group_col = normalize_group_col(df, _state.get("group_col"))
    value_cols = _state.get("value_cols")

    if df is None or id_col is None or not value_cols or len(value_cols) < 2:
        with status:
            print("解析に必要な列が不足しています。")
            print("検体ID列と、2列以上の比較列が必要です。")
        return

    if mode_dd.value == "adjacent":
        pairs = list(zip(value_cols[:-1], value_cols[1:]))

    elif mode_dd.value == "all":
        pairs = list(itertools.combinations(value_cols, 2))

    else:
        bases = list(baseline_sel.value) if baseline_sel.value else [value_cols[0]]
        pairs = []

        for b0 in bases:
            for c in value_cols:
                if c != b0:
                    pairs.append((b0, c))

    methods = REGRESSION_METHODS_ALL if all_reg_ck.value else [reg_dd.value]

    lam = deming_lambda.value

    lo_range = float(range_min_txt.value)
    hi_range = float(range_max_txt.value)

    if lo_range > hi_range:
        lo_range, hi_range = hi_range, lo_range

    try:
        dirs = make_output_dirs(OUTPUT_ROOT)
    except Exception as e:
        with status:
            print("出力先フォルダ作成エラー:", repr(e))
        return

    out_dir = dirs["plots"]
    excel_dir = dirs["excel"]

    summary_rows = []
    group_rows = []
    outlier_tables = []
    sample_metric_tables = []
    img_paths = []
    shown = 0

    for method in methods:
        for xcol, ycol in pairs:
            df_pair = apply_value_range_filter(
                df,
                xcol=xcol,
                ycol=ycol,
                use_range=use_range_ck.value,
                lo=lo_range,
                hi=hi_range,
                mode=range_mode_dd.value
            )

            sub = df_pair[[xcol, ycol]].dropna()

            if len(sub) < 2:
                continue

            x = pd.to_numeric(sub[xcol], errors="coerce").astype(float).to_numpy()
            y = pd.to_numeric(sub[ycol], errors="coerce").astype(float).to_numpy()

            # 重要：
            # ここで計算する r, slope, intercept は、
            # zMADで後から抽出される乖離候補を除外せず、
            # 欠損除外・対象範囲フィルタ後の全データを用いて算出する。
            r = pearson_r(x, y)

            a, b, fit_info = regression_fit_info(
                x,
                y,
                method=method,
                deming_lambda=lam
            )

            fig, used_sub, flagged, bias, loa = plot_suite(
                df=df_pair,
                id_col=id_col,
                group_col=group_col,
                xcol=xcol,
                ycol=ycol,
                method=method,
                lam=lam,
                a=a,
                b=b,
                r=r,
                fit_info=fit_info,
                z_thresh=z_thresh.value,
                outlier_label_top=label_top.value,
                add_diag=add_diag_ck.value,
                show_fit=show_fit_ck.value,
                show_stats=show_stats_ck.value,
                show_equation=show_equation_ck.value,
                show_ci=show_ci_ck.value,
                show_ba_text=show_ba_text_ck.value,
                show_outlier_text=show_outlier_text_ck.value,
                normal_s=normal_point_size.value,
                outlier_s=outlier_point_size.value,
                outlier_lw=outlier_line_width.value,
                fig_width=fig_width_slider.value,
                fig_height=fig_height_slider.value,
                dpi=plot_dpi_slider.value
            )

            if fig is None:
                continue

            if show_py_ck.value and max_show.value > 0 and shown < max_show.value:
                with plots_out:
                    display(fig)

                shown += 1

            png = out_dir / f"QC_{safe_name(method)}_{safe_name(ycol)}_vs_{safe_name(xcol)}.png"

            fig.savefig(
                png,
                bbox_inches="tight"
            )

            plt.close(fig)

            img_paths.append(png)

            n_out = int(len(flagged)) if flagged is not None else 0

            summary_rows.append({
                "regression": method,
                "regression_label": REGRESSION_LABELS.get(method, method),
                "X": xcol,
                "Y": ycol,
                "n": len(x),
                "r": r,
                "|r|": abs(r) if np.isfinite(r) else np.nan,
                "slope": a,
                "intercept": b,
                "lambda": lam if method == "Deming" else np.nan,
                "slope_95CI_low": fit_info.get("slope_ci_low", np.nan),
                "slope_95CI_high": fit_info.get("slope_ci_high", np.nan),
                "intercept_95CI_low": fit_info.get("intercept_ci_low", np.nan),
                "intercept_95CI_high": fit_info.get("intercept_ci_high", np.nan),
                "PB_n_slopes": fit_info.get("pb_n_slopes", np.nan),
                "PB_CI_method": fit_info.get("pb_ci_method", ""),
                "BA_bias(y-x)": bias,
                "BA_LoA_low": loa[0],
                "BA_LoA_high": loa[1],
                "n_outliers": n_out,
                "outlier_z_thresh": z_thresh.value,
                "outlier_basis": "回帰残差 y-(slope*x+intercept) をMADで標準化",
                "outlier_note": "乖離候補であり、自動除外ではない",
                "outlier_excluded_from_fit": False,
                "fit_data_note": "相関係数・回帰係数は乖離候補を除外せず、解析対象データ全体で算出",
                "range_filter_used": bool(use_range_ck.value),
                "range_min": lo_range if use_range_ck.value else np.nan,
                "range_max": hi_range if use_range_ck.value else np.nan,
                "range_condition": range_mode_dd.value if use_range_ck.value else "",
                "normal_point_size": normal_point_size.value,
                "outlier_point_size": outlier_point_size.value,
                "outlier_line_width": outlier_line_width.value,
                "figure_width": fig_width_slider.value,
                "figure_height": fig_height_slider.value,
                "plot_dpi": plot_dpi_slider.value,
                "output_root": str(dirs["root"])
            })

            gdf = groupwise_stats(
                df_pair,
                id_col,
                group_col,
                xcol,
                ycol,
                method,
                lam
            )

            if not gdf.empty:
                group_rows.append(gdf)

            if flagged is not None and not flagged.empty:
                outlier_tables.append(flagged.assign(
                    regression=method,
                    regression_label=REGRESSION_LABELS.get(method, method),
                    slope=a,
                    intercept=b,
                    r=r
                ))

            metrics_df = compute_pair_sample_metrics(
                df=df_pair,
                id_col=id_col,
                group_col=group_col,
                xcol=xcol,
                ycol=ycol,
                a=a,
                b=b
            )

            if not metrics_df.empty:
                metrics_df["regression"] = method
                metrics_df["regression_label"] = REGRESSION_LABELS.get(method, method)
                metrics_df["slope"] = a
                metrics_df["intercept"] = b
                metrics_df["r"] = r
                metrics_df["slope_95CI_low"] = fit_info.get("slope_ci_low", np.nan)
                metrics_df["slope_95CI_high"] = fit_info.get("slope_ci_high", np.nan)
                metrics_df["intercept_95CI_low"] = fit_info.get("intercept_ci_low", np.nan)
                metrics_df["intercept_95CI_high"] = fit_info.get("intercept_ci_high", np.nan)
                metrics_df["outlier_z_thresh"] = z_thresh.value
                metrics_df["outlier_excluded_from_fit"] = False
                metrics_df["fit_data_note"] = "相関係数・回帰係数は乖離候補を除外せず、解析対象データ全体で算出"
                metrics_df["range_filter_used"] = bool(use_range_ck.value)
                metrics_df["range_min"] = lo_range if use_range_ck.value else np.nan
                metrics_df["range_max"] = hi_range if use_range_ck.value else np.nan
                metrics_df["range_condition"] = range_mode_dd.value if use_range_ck.value else ""

                sample_metric_tables.append(metrics_df)

    if not summary_rows:
        with status:
            print("作図・解析できるペアがありませんでした。")
            print("欠損が多い、対象範囲が狭すぎる、または比較列が不足している可能性があります。")
        return

    summary_df = pd.DataFrame(summary_rows)

    if best_mode.value == "abs":
        best = summary_df.sort_values("|r|", ascending=False).iloc[0]
        best_label = "|r|最大"
    else:
        best = summary_df.sort_values("r", ascending=False).iloc[0]
        best_label = "r最大（正相関優先）"

    with status:
        print("=== 実行結果 ===")
        print("入力:", xlsx_path)
        print("出力root:", dirs["root"])
        print("モード:", mode_dd.value, " / ペア数:", len(pairs))
        print(
            "回帰法:",
            "全回帰法" if all_reg_ck.value else REGRESSION_LABELS.get(reg_dd.value, reg_dd.value)
        )
        print(
            f"ベスト相関（{best_label}）: "
            f"{best['regression_label']} / {best['Y']} vs {best['X']}"
        )
        print(
            f"  n={int(best['n'])}  "
            f"r={best['r']:.4f}  "
            f"slope={best['slope']:.4f}  "
            f"intercept={best['intercept']:.4f}"
        )
        print(
            f"  BA bias={best['BA_bias(y-x)']:.4g}  "
            f"LoA=[{best['BA_LoA_low']:.4g}, {best['BA_LoA_high']:.4g}]  "
            f"outliers={int(best['n_outliers'])}"
        )
        print("※相関係数・回帰係数は、乖離候補を除外せずに算出しています。")
        print(f"Python表示枚数: {shown}")

    output_xlsx = excel_dir / f"{xlsx_path.stem}{OUT_SUFFIX}{xlsx_path.suffix}"

    insert_images_into_excel(
        input_xlsx=xlsx_path,
        output_xlsx=output_xlsx,
        image_paths=img_paths,
        plot_sheet=SHEET_PLOTS,
        start_cell="A1",
        row_step=44
    )

    # summary
    summary_to_write = summary_df.drop(columns=["|r|"], errors="ignore")
    write_df_to_sheet(output_xlsx, summary_to_write, SHEET_SUMMARY)

    # group stats
    if group_rows:
        gs = pd.concat(group_rows, axis=0, ignore_index=True)
        write_df_to_sheet(output_xlsx, gs, SHEET_GROUP_STATS)
    else:
        empty_gs = pd.DataFrame([{
            "note": "有効な種別列がない、または群別統計を作成できるデータがありません。"
        }])
        write_df_to_sheet(output_xlsx, empty_gs, SHEET_GROUP_STATS)

    # outliers
    if outlier_tables:
        odf = pd.concat(outlier_tables, axis=0, ignore_index=True)

        cols_first = ["regression", "regression_label", "X", "Y", id_col]

        if group_col:
            cols_first.append(group_col)

        cols_first += [
            "slope",
            "intercept",
            "r",
            "diff_y_minus_x",
            "abs_diff_yx",
            "mean_xy",
            "rel_diff_yx_pct",
            "bias_pct_vs_x",
            "ratio_y_over_x",
            "pred_y",
            "residual",
            "abs_residual",
            "z_MAD",
            "abs_z_MAD",
            "outlier_level",
            "outlier_basis",
            "outlier_note"
        ]

        cols_first = [c for c in cols_first if c in odf.columns]
        rest_cols = [c for c in odf.columns if c not in cols_first]

        odf = odf[cols_first + rest_cols]

        write_df_to_sheet(output_xlsx, odf, SHEET_OUTLIERS)
    else:
        empty_out = pd.DataFrame([{
            "note": "指定した zMAD 閾値以上の乖離候補はありませんでした。"
        }])
        write_df_to_sheet(output_xlsx, empty_out, SHEET_OUTLIERS)

    # sample metrics
    if sample_metric_tables:
        smdf = pd.concat(sample_metric_tables, axis=0, ignore_index=True)

        cols_first = ["regression", "regression_label", "X", "Y", id_col]

        if group_col:
            cols_first.append(group_col)

        metric_cols = [
            "slope",
            "intercept",
            "r",
            "slope_95CI_low",
            "slope_95CI_high",
            "intercept_95CI_low",
            "intercept_95CI_high",
            "diff_y_minus_x",
            "abs_diff_yx",
            "mean_xy",
            "rel_diff_yx_pct",
            "bias_pct_vs_x",
            "ratio_y_over_x",
            "pred_y",
            "residual",
            "abs_residual",
            "z_MAD",
            "abs_z_MAD",
            "outlier_level",
            "outlier_z_thresh",
            "outlier_basis",
            "outlier_note",
            "outlier_excluded_from_fit",
            "fit_data_note",
            "range_filter_used",
            "range_min",
            "range_max",
            "range_condition"
        ]

        cols_first += metric_cols
        cols_first = [c for c in cols_first if c in smdf.columns]

        rest_cols = [c for c in smdf.columns if c not in cols_first]

        smdf = smdf[cols_first + rest_cols]

        write_df_to_sheet(output_xlsx, smdf, SHEET_SAMPLE_METRICS)

    # metric definitions
    metric_def_df = make_metric_definitions_df()
    write_df_to_sheet(output_xlsx, metric_def_df, SHEET_METRIC_DEFINITIONS)

    with status:
        print("\n完了！")
        print("出力Excel:", output_xlsx)
        print("画像フォルダ:", dirs["plots"])
        print("表フォルダ:", dirs["tables"])
        print("ログフォルダ:", dirs["logs"])
        print(
            f"シート: {SHEET_PLOTS}, {SHEET_SUMMARY}, {SHEET_GROUP_STATS}, "
            f"{SHEET_OUTLIERS}, {SHEET_SAMPLE_METRICS}, {SHEET_METRIC_DEFINITIONS}"
        )
        print("※乖離候補は自動除外していません。")
        print("※r/slope/interceptは、欠損除外・対象範囲フィルタ後の全データで算出しています。")


run_btn.on_click(run_all)